# 3D_demo v2 — Video phòng ➜ COLMAP ➜ 3DGS (MCMC + Kill-then-Relocate + Bilateral grid + Depth prior) ➜ .ply / .splat
**Chạy lần lượt từ trên xuống.** Cần GPU (Colab: *Runtime → Change runtime type → GPU*; A100/L4 khuyên dùng).

| Mục | Làm gì | Hay chỉnh |
|---|---|---|
| 0. Setup | Cài đặt + cấu hình | `SCENE_NAME`, `MAX_STEPS`, `FRAME_HEIGHT` |
| 1A / 1B | Lấy video (máy / Drive) → chọn frame NÉT NHẤT theo đoạn, bỏ frame trùng | `INPUT`, `TARGET_FRAMES` |
| 2 | COLMAP (sequential cho video) + khử méo + kiểm tra chất lượng | `COLMAP_MATCHER` |
| 3A | Cài **gsplat** (thử v1.5.3, lỗi thì tự lùi về v1.3.0) + chạy thử GPU | — |
| 3B | **Depth prior**: Depth Anything V2 cho từng ảnh + tự kiểm tra độ tin cậy bằng điểm COLMAP | `USE_MONO_DEPTH`, `MONO_MODEL` |
| 3C | Train: MCMC + scene scale bền vững + **Kill-then-Relocate** + **Bilateral grid** + **Depth prior** | `USE_KR`, `USE_BILAGRID`, `USE_MONO` |
| 4 | Lọc floater 7 bước, xuất `scene.ply` / `scene.splat` / `removed.ply` | `FILTER` |
| 4B | **Lấy file .ply**: `<SCENE_NAME>.ply` (+ bản lite) → Drive + tải về máy | `DOWNLOAD_TO_PC`, `MAKE_LITE` |
| 5 | Web viewer | — |
| 6 | *(tuỳ chọn)* **Ablation**: đo đóng góp từng cải tiến → bảng Markdown/CSV cho báo cáo | `RUN_ABLATION` |


In [ ]:
# ── 0. Setup & cấu hình ─────────────────────────────────────────────────────
import os, sys, shutil, glob, struct, json, subprocess
import torch

IN_COLAB = "google.colab" in sys.modules      # tự nhận Colab / RunPod

assert torch.cuda.is_available(), "Không thấy GPU (Colab: Runtime > Change runtime type > GPU)"
print("GPU:", torch.cuda.get_device_name(0), "| PyTorch", torch.__version__, "| CUDA", torch.version.cuda)

!apt-get update -qq
!DEBIAN_FRONTEND=noninteractive apt-get install -y -qq ffmpeg colmap xvfb libgl1 libglib2.0-0
!pip install -q ninja scipy
if not IN_COLAB:
    !pip install -q opencv-python-headless

import numpy as np, cv2
print("nvcc:", shutil.which("nvcc") or "KHÔNG THẤY -> cần template PyTorch '-devel' để compile gsplat")

# ── CẤU HÌNH (đổi SCENE_NAME cho mỗi video / bộ ảnh) ─────────────────────────
SCENE_NAME     = "scene1"
FRAME_HEIGHT   = 1080         # cạnh NGẮN tối đa của frame/ảnh sau chuẩn hoá (px)
COLMAP_MATCHER = "auto"       # "auto" (video -> sequential, ảnh -> exhaustive) | "sequential" | "exhaustive"
MAX_STEPS      = 7_000        # số bước train 3DGS (7000: nhanh, A100 ~5–10 phút; muốn đẹp hơn: 15_000–30_000)
DATA_FACTOR    = 1            # 1 = train ở đúng độ phân giải frame đã chuẩn hoá

BASE          = "/content" if IN_COLAB else "/workspace"   # Colab: /content (mất khi hết phiên)
WORK_DIR      = f"{BASE}/scene3d_work/{SCENE_NAME}"
FRAMES_DIR    = f"{WORK_DIR}/frames"
COLMAP_DIR    = f"{WORK_DIR}/colmap"
GSPLAT_REPO   = f"{BASE}/gsplat"
GSPLAT_OUTPUT = f"{WORK_DIR}/gsplat_output"
OUT_DIR       = f"{BASE}/outputs/{SCENE_NAME}"
VIEWER_DIR    = f"{BASE}/viewer/{SCENE_NAME}"
DRIVE_BACKUP  = f"/content/drive/MyDrive/3D_outputs/{SCENE_NAME}"   # sao lưu checkpoint + kết quả (nếu đã mount Drive)
for d in (WORK_DIR, OUT_DIR, VIEWER_DIR): os.makedirs(d, exist_ok=True)

# Dữ liệu đầu vào (VIDEO hoặc ZIP ảnh, từ máy hoặc Google Drive): xem mục 1 bên dưới.


---
## 1. Nhập dữ liệu — VIDEO (.mp4/.mov) hoặc ZIP ảnh
**1A. Lấy dữ liệu.** Điền `INPUT` rồi chạy cell:
- `"/content/drive/MyDrive/data/video2.mp4"` → video trong Google Drive (tự mount Drive)
- `""` → upload video / `.zip` từ máy (hiện nút *Choose files*; file > 500 MB nên để trên Drive)
- `"https://drive.google.com/file/d/<ID>/view..."` → link Drive

Cell nhận diện định dạng thật của file (video / zip / trang HTML lỗi), xoá bản cũ hỏng, và **chạy lại không cần tải lại** (`FORCE_NEW_INPUT = True` để đổi dữ liệu).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── 1A. NHẬP DỮ LIỆU: VIDEO (.mp4/.mov) hoặc ZIP ảnh — từ máy hoặc Google Drive ──
# Điền INPUT theo MỘT trong các dạng sau rồi chạy cell:
#   ""                                             -> upload video / .zip từ máy (hiện nút "Choose files")
#   "/content/drive/MyDrive/data/video2.mp4"       -> VIDEO trong Google Drive (tự mount Drive)
#   "/content/drive/MyDrive/data/phong.zip"        -> file .zip trong Google Drive (tự mount Drive)
#   "/content/drive/MyDrive/data/anh_phong"        -> THƯ MỤC ảnh trong Google Drive (không cần nén)
#   "https://drive.google.com/file/d/<ID>/view..." -> link Drive của file .zip (file riêng tư vẫn được,
#                                                     miễn tài khoản Google đăng nhập Colab có quyền xem)
#   "https://drive.google.com/drive/folders/<ID>"  -> link thư mục ảnh trên Drive
# Hỗ trợ: video mp4/mov/mkv/avi/webm ; .zip/.7z/.rar/.tar(.gz) ; ảnh jpg, png, bmp, tif, webp, heic (iPhone).
# Mẹo: file > 500 MB nên để trên Drive thay vì upload (upload qua trình duyệt chậm, dễ đứt).
INPUT = "/content/drive/MyDrive/data/video2.mp4"
FORCE_NEW_INPUT = False   # True = bỏ dữ liệu đã nạp lần trước, lấy lại từ nguồn

import os, re, sys, glob, shutil, tempfile, subprocess

INPUT_DIR  = f"{WORK_DIR}/input"            # nơi giữ bản sao file zip đã lấy về
RECORD     = f"{INPUT_DIR}/_source.txt"     # ghi nhớ nguồn để chạy lại không phải upload lại
ARCHIVES   = ("zip", "7z", "rar", "tar", "gzip")
USABLE     = ARCHIVES + ("folder", "video")
os.makedirs(INPUT_DIR, exist_ok=True)


def sniff(path):
    """Nhận diện định dạng THẬT của file dựa vào các byte đầu (không tin đuôi file)."""
    if os.path.isdir(path):
        return "folder"
    with open(path, "rb") as f:
        h = f.read(512)
    if h[4:8] == b"ftyp":                                        return "video"   # mp4 / mov / m4v / 3gp
    if h[:4] == b"\x1a\x45\xdf\xa3":                             return "video"   # mkv / webm
    if h[:4] == b"RIFF" and h[8:12] == b"AVI ":                   return "video"   # avi
    if h[:4] in (b"PK\x03\x04", b"PK\x05\x06", b"PK\x07\x08"): return "zip"
    if h[:6] == b"7z\xbc\xaf\x27\x1c":                          return "7z"
    if h[:4] == b"Rar!":                                         return "rar"
    if h[:2] == b"\x1f\x8b":                                     return "gzip"
    if len(h) > 262 and h[257:262] == b"ustar":                  return "tar"
    if h.lstrip()[:1] == b"<" and (b"html" in h.lower() or b"<!doctype" in h.lower()): return "html"
    return "unknown"


def _safe(name):
    name = os.path.basename(name.strip().rstrip("/")) or "input"
    return re.sub(r"[^\w.\-]+", "_", name)


def _mb(p):
    return os.path.getsize(p) / 1e6


def _drive_id(s):
    """Lấy ID từ link Google Drive (hoặc chính chuỗi ID)."""
    for pat in (r"/file/d/([A-Za-z0-9_-]{20,})", r"/folders/([A-Za-z0-9_-]{20,})",
                r"[?&]id=([A-Za-z0-9_-]{20,})"):
        m = re.search(pat, s)
        if m:
            return m.group(1)
    if re.fullmatch(r"[A-Za-z0-9_-]{25,}", s.strip()):
        return s.strip()
    return None


def _normalize(s):
    """Sửa các cách viết đường dẫn hay gặp: thiếu '/' đầu, dư dấu nháy/khoảng trắng, 'My Drive'."""
    s = s.strip().strip('"').strip("'").strip()
    if s == "" or s.startswith(("http://", "https://")) or _drive_id(s) and not ("/" in s or "." in s):
        return s
    s = s.replace("\\", "/")
    s = re.sub(r"^\./", "", s)
    for pre, fix in (("content/", "/content/"), ("drive/MyDrive/", "/content/drive/MyDrive/"),
                     ("MyDrive/", "/content/drive/MyDrive/"), ("My Drive/", "/content/drive/MyDrive/"),
                     ("/MyDrive/", "/content/drive/MyDrive/"), ("/drive/MyDrive/", "/content/drive/MyDrive/")):
        if s.startswith(pre):
            return fix + s[len(pre):]
    s = s.replace("/content/drive/My Drive/", "/content/drive/MyDrive/")
    if not s.startswith("/") and os.path.exists(os.path.abspath(s)):
        return os.path.abspath(s)
    return s


def _ensure_drive():
    if not os.path.isdir("/content/drive/MyDrive"):
        from google.colab import drive
        drive.mount("/content/drive")


def from_upload():
    """Upload từ máy. Ghi thẳng nội dung vừa upload ra file mới (không đụng file cũ trùng tên)."""
    if not IN_COLAB:
        raise RuntimeError(f"Không ở Colab: hãy chép file zip lên máy rồi đặt INPUT = đường dẫn file đó.")
    from google.colab import files
    tmp, cwd = tempfile.mkdtemp(prefix="upload_"), os.getcwd()
    os.chdir(tmp)                      # Colab lưu file upload vào thư mục hiện tại -> dùng thư mục tạm
    try:
        print("Bấm 'Choose files' và chọn video (.mp4/.mov) hoặc file .zip ảnh. Chờ đến khi hiện '100% done'...")
        up = files.upload()
    finally:
        os.chdir(cwd)
    if not up:
        shutil.rmtree(tmp, ignore_errors=True)
        raise RuntimeError("Chưa chọn file nào. Chạy lại cell và chọn video hoặc file .zip.")
    name, data = next(iter(up.items()))
    dest = f"{INPUT_DIR}/{_safe(name)}"
    with open(dest, "wb") as f:
        f.write(data)
    shutil.rmtree(tmp, ignore_errors=True)
    if len(data) == 0:
        raise RuntimeError("File upload rỗng (0 byte) — upload bị gián đoạn. Thử lại hoặc để file trên Drive.")
    return dest


def from_path(p):
    """Đường dẫn trên Drive (đã mount) hoặc trên máy chạy notebook (RunPod...)."""
    if p.startswith("/content/drive"):
        _ensure_drive()
    if not os.path.exists(p):
        msg = f"Không thấy: {p}"
        if p.startswith("/content/drive"):
            base = os.path.basename(p.rstrip("/"))
            r = subprocess.run(["find", "/content/drive/MyDrive", "-maxdepth", "4", "-iname", f"*{base}*"],
                               capture_output=True, text=True, timeout=180)
            cands = [c for c in r.stdout.splitlines() if c][:10]
            if not cands:
                r = subprocess.run(["find", "/content/drive/MyDrive", "-maxdepth", "3", "(", "-iname", "*.zip", "-o", "-iname", "*.mp4", "-o", "-iname", "*.mov", ")"],
                                   capture_output=True, text=True, timeout=180)
                cands = [c for c in r.stdout.splitlines() if c][:15]
                msg += "\nCác file video/.zip tìm thấy trong MyDrive:" if cands else "\nKhông thấy video/.zip nào trong MyDrive (độ sâu 3)."
            else:
                msg += "\nCó thể bạn muốn dùng một trong các đường dẫn sau:"
            msg += "".join(f"\n   {c}" for c in cands)
            msg += ("\n(File nằm trong 'Shared with me' sẽ không hiện ở đây: chuột phải -> 'Add shortcut to Drive', "
                    "hoặc dán link Drive vào INPUT.)")
        raise FileNotFoundError(msg)
    if os.path.isdir(p):
        return p                        # thư mục ảnh: dùng trực tiếp, không sao chép, không xoá
    dest = f"{INPUT_DIR}/{_safe(p)}"
    print(f"Đang chép {p} ({_mb(p):.1f} MB) về máy Colab...")
    shutil.copyfile(p, dest)            # chép về ổ local -> giải nén nhanh hơn đọc trực tiếp từ Drive
    return dest


def from_drive_id(fid):
    """Tải bằng Google Drive API dưới tài khoản của bạn (file riêng tư / Shared with me vẫn được)."""
    if not IN_COLAB:
        raise RuntimeError("Tải bằng link Drive chỉ hỗ trợ trên Colab. Trên máy khác hãy tải file về rồi dùng đường dẫn.")
    from google.colab import auth
    auth.authenticate_user()
    from googleapiclient.discovery import build
    from googleapiclient.errors import HttpError
    from googleapiclient.http import MediaIoBaseDownload
    svc = build("drive", "v3", cache_discovery=False)

    def download(file_id, dest, size=None):
        req = svc.files().get_media(fileId=file_id, supportsAllDrives=True)
        with open(dest, "wb") as fh:
            dl, done = MediaIoBaseDownload(fh, req, chunksize=64 * 1024 * 1024), False
            while not done:
                status, done = dl.next_chunk()
                if status and size and int(size) > 64e6:
                    print(f"   {status.progress() * 100:5.1f}%", end="\r")
        return dest

    try:
        meta = svc.files().get(fileId=fid, fields="id,name,size,mimeType", supportsAllDrives=True).execute()
    except HttpError as e:
        raise RuntimeError(f"Không mở được file Drive ID={fid} ({e.status_code if hasattr(e, 'status_code') else e}).\n"
                           "-> Kiểm tra: link đúng chưa, và tài khoản Google vừa đăng nhập có quyền xem file không "
                           "(mở link bằng đúng tài khoản đó trên trình duyệt).") from None
    print(f"Drive: {meta['name']} | {meta['mimeType']} | {int(meta.get('size', 0)) / 1e6:.1f} MB")

    if meta["mimeType"] == "application/vnd.google-apps.folder":
        out = f"{INPUT_DIR}/drive_folder_{fid[:8]}"
        os.makedirs(out, exist_ok=True)
        items, token = [], None
        while True:
            res = svc.files().list(q=f"'{fid}' in parents and trashed=false", pageToken=token, pageSize=1000,
                                   fields="nextPageToken, files(id,name,mimeType,size)",
                                   supportsAllDrives=True, includeItemsFromAllDrives=True).execute()
            items += res.get("files", [])
            token = res.get("nextPageToken")
            if not token:
                break
        items = [it for it in items if not it["mimeType"].startswith("application/vnd.google-apps")]
        if not items:
            raise RuntimeError("Thư mục Drive không có file nào tải được (chỉ có thư mục con / Google Docs).")
        print(f"Tải {len(items)} file từ thư mục Drive...")
        for i, it in enumerate(items, 1):
            download(it["id"], f"{out}/{_safe(it['name'])}")
            if i % 25 == 0 or i == len(items):
                print(f"   {i}/{len(items)}")
        return out
    if meta["mimeType"].startswith("application/vnd.google-apps"):
        raise RuntimeError(f"Link trỏ tới {meta['mimeType']} (Google Docs/Sheets...), không phải file .zip.")
    dest = f"{INPUT_DIR}/{_safe(meta['name'])}"
    print("Đang tải về Colab...")
    download(fid, dest, meta.get("size"))
    print()
    return dest


# ── Dùng lại dữ liệu đã nạp nếu nguồn không đổi (chạy lại cell không phải upload lại) ──
DATA_SRC = None
if not FORCE_NEW_INPUT and os.path.exists(RECORD):
    prev_input, prev_path = (open(RECORD, encoding="utf-8").read().split("\n") + ["", ""])[:2]
    if prev_input == INPUT and os.path.exists(prev_path) and sniff(prev_path) in USABLE:
        DATA_SRC = prev_path
        print(f"Dùng lại dữ liệu đã nạp: {DATA_SRC}   (đặt FORCE_NEW_INPUT = True để lấy lại)")

if DATA_SRC is None:
    # Xoá bản cũ (kể cả bản hỏng của lần trước) -> không bao giờ dính file lỗi cũ
    for p in glob.glob(f"{INPUT_DIR}/*"):
        shutil.rmtree(p, ignore_errors=True) if os.path.isdir(p) else os.remove(p)
    src = _normalize(INPUT)
    if src != INPUT.strip():
        print(f"(Đã hiểu INPUT là: {src})")
    if src == "":
        DATA_SRC = from_upload()
    elif os.path.exists(src) or src.startswith("/"):
        DATA_SRC = from_path(src)
    elif _drive_id(src):
        DATA_SRC = from_drive_id(_drive_id(src))
    else:
        raise ValueError(f"Không hiểu INPUT = {INPUT!r}.\nVí dụ đúng: \"\" (upload từ máy), "
                         "\"/content/drive/MyDrive/data/room.zip\", hoặc link https://drive.google.com/...")

# ── Kiểm tra định dạng thật ──────────────────────────────────────────────────
DATA_KIND = sniff(DATA_SRC)
if DATA_KIND == "html":
    head = open(DATA_SRC, "rb").read(300)
    os.remove(DATA_SRC)
    raise RuntimeError("File nhận được là TRANG WEB (HTML), không phải zip — thường do tải link Drive chưa có quyền.\n"
                       f"Đầu file: {head[:200]!r}\n-> Dùng đường dẫn /content/drive/... hoặc dán link Drive vào INPUT.")
if DATA_KIND not in USABLE:
    head = open(DATA_SRC, "rb").read(16)
    raise RuntimeError(f"File {os.path.basename(DATA_SRC)} không phải video hay file nén (16 byte đầu: {head!r}).\n"
                       "-> Dùng video .mp4/.mov hoặc nén thư mục ảnh thành .zip, rồi chạy lại cell (FORCE_NEW_INPUT = True).")

with open(RECORD, "w", encoding="utf-8") as f:
    f.write(f"{INPUT}\n{DATA_SRC}")
size_txt = "thư mục" if DATA_KIND == "folder" else f"{_mb(DATA_SRC):.1f} MB"
print(f"✓ Dữ liệu: {DATA_SRC}  | định dạng: {DATA_KIND} | {size_txt}")


**1B. Chọn frame.** Video được chia thành `TARGET_FRAMES` đoạn; mỗi đoạn lấy frame **nét nhất** trong ~4 ứng viên liền kề (thay vì trích đều 5 fps rồi bỏ 10% mờ). Bỏ frame vẫn mờ, bỏ frame trùng lúc camera đứng yên, tự xoay video dọc, tone-map video HDR của iPhone. Với zip ảnh: xoay EXIF, bỏ ảnh hỏng/khác hướng, đưa về cùng kích thước, bỏ ảnh mờ.

In [ ]:
# ── 1B. CHUẨN BỊ FRAME -> FRAMES_DIR (đầu vào của COLMAP) ──────────────────
# VIDEO: chia video thành TARGET_FRAMES đoạn liền nhau, mỗi đoạn lấy frame NÉT NHẤT trong ~CAND_PER_PICK ứng viên,
#        bỏ frame vẫn mờ và frame gần như trùng (lúc camera đứng yên). Tốt hơn trích đều N fps rồi bỏ 10% mờ.
# ZIP / THƯ MỤC ẢNH: giải nén, xoay EXIF, bỏ ảnh hỏng/khác hướng, đưa về cùng kích thước, bỏ BLUR_DROP_PCT% ảnh mờ.
TARGET_FRAMES = 250    # (video) số frame muốn giữ: 150–300 là hợp với COLMAP + 3DGS cho 1 căn phòng
CAND_PER_PICK = 4      # (video) mỗi frame giữ lại được chọn từ ~4 frame liền kề -> chọn cái nét nhất
MIN_SHARP_REL = 0.35   # (video) bỏ frame đã chọn nhưng vẫn mờ (< 35% độ nét trung vị)
DUP_DIFF      = 1.5    # (video) bỏ frame gần như y hệt frame trước (camera đứng yên); 0 = tắt
TRIM_START    = 0.0    # (video) bỏ X giây đầu (lúc vừa bấm quay)
TRIM_END      = 0.0    # (video) bỏ X giây cuối (lúc hạ máy)
BLUR_DROP_PCT = 10     # (ảnh) bỏ % ảnh mờ nhất; 0 = không lọc. Với video không dùng (đã chọn frame nét ở trên)
MAX_IMAGES    = 0      # (ảnh) 0 = dùng hết; vd 300 = lấy đều 300 ảnh

import os, re, sys, json, math, glob, shutil, subprocess, zipfile, tarfile
from collections import Counter
import numpy as np, cv2

assert "DATA_SRC" in globals() and os.path.exists(DATA_SRC), "Chưa có dữ liệu — chạy cell 1A trước."

IMG_EXT   = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp", ".heic", ".heif"}
VID_EXT   = {".mp4", ".mov", ".mkv", ".avi", ".webm", ".m4v"}
EXTRACT   = f"{WORK_DIR}/extracted"       # thư mục giải nén tạm (xoá khi xong)
VFRAMES   = f"{WORK_DIR}/video_frames"    # frame ứng viên trích từ video (xoá khi xong)


def _run(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True)


def _ensure_7z():
    if not shutil.which("7z"):
        print("Cài p7zip (7z) ...")
        _run("apt-get install -y -qq p7zip-full p7zip-rar || apt-get install -y -qq p7zip-full")
    if not shutil.which("7z"):
        raise RuntimeError("Không cài được 7z. Hãy nén lại thành .zip chuẩn.")


def extract(src, dst):
    """Giải nén zip/7z/rar/tar vào dst. zip lỗi với Python (vd nén Deflate64 trên Windows) -> thử 7z."""
    kind = sniff(src)
    shutil.rmtree(dst, ignore_errors=True)
    os.makedirs(dst)
    if kind == "zip":
        try:
            with zipfile.ZipFile(src) as z:
                z.extractall(dst)
            return
        except (zipfile.BadZipFile, NotImplementedError, OSError, EOFError) as e:
            print(f"Python zipfile không giải nén được ({type(e).__name__}: {e}) -> thử 7z")
            shutil.rmtree(dst, ignore_errors=True); os.makedirs(dst)
    elif kind in ("tar", "gzip"):
        try:
            with tarfile.open(src) as t:
                try:
                    t.extractall(dst, filter="data")
                except TypeError:
                    t.extractall(dst)
            return
        except tarfile.TarError as e:
            raise RuntimeError(f"Không giải nén được file tar/gz: {e}")
    _ensure_7z()
    r = _run(f'7z x -y -o"{dst}" "{src}"')
    if r.returncode != 0:
        n_ok = sum(len(fs) for _, _, fs in os.walk(dst))
        if n_ok > 0:      # file nén bị cụt (upload/tải chưa xong): vẫn cứu phần đã giải nén được
            print(f"⚠ File nén bị THIẾU PHẦN CUỐI (upload/tải lên Drive chưa xong). Cứu được {n_ok} file; "
                  "ảnh cuối bị cụt sẽ tự bị bỏ qua. Nên upload lại file zip đầy đủ khi có thể.")
            return
        raise RuntimeError("Giải nén thất bại — file bị hỏng hoặc chưa tải/upload đủ.\n"
                           f"{(r.stdout + r.stderr)[-600:]}\n-> Chạy lại cell 1A với FORCE_NEW_INPUT = True.")


def list_files(root, exts):
    out = []
    for r, dirs, fs in os.walk(root):
        dirs[:] = [d for d in dirs if not d.startswith(".") and d != "__MACOSX"]
        for f in fs:
            if f.startswith(".") or f.startswith("._"):
                continue
            if os.path.splitext(f)[1].lower() in exts:
                out.append(os.path.join(r, f))
    natural = lambda p: [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", os.path.relpath(p, root))]
    return sorted(out, key=natural)     # IMG_2 trước IMG_10



def _sharpness(path):
    g = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if g is None:
        return 0.0
    g = cv2.resize(g, (640, max(1, round(640 * g.shape[0] / g.shape[1]))), interpolation=cv2.INTER_AREA)
    return float(cv2.Laplacian(g, cv2.CV_64F).var())


def _thumb(path):
    g = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    return cv2.resize(g, (64, 36), interpolation=cv2.INTER_AREA).astype(np.float32)


def select_video_frames(video):
    """Trích frame ứng viên bằng ffmpeg rồi chọn frame nét nhất trong từng đoạn. Trả về list đường dẫn."""
    r = subprocess.run(["ffprobe", "-v", "error", "-select_streams", "v:0", "-show_entries",
                        "stream=width,height,avg_frame_rate,color_transfer:format=duration", "-of", "json", video],
                       capture_output=True, text=True)
    info = json.loads(r.stdout or "{}")
    st = (info.get("streams") or [{}])[0]
    try:
        dur = float(info.get("format", {}).get("duration", 0))
    except ValueError:
        dur = 0.0
    num, _, den = str(st.get("avg_frame_rate", "30/1")).partition("/")
    src_fps = float(num) / float(den or 1) if float(den or 1) else 30.0
    if dur <= 0:
        raise RuntimeError(f"Không đọc được độ dài video (ffprobe: {r.stderr[-300:]}). File video có thể bị hỏng.")
    use = max(0.5, dur - TRIM_START - TRIM_END)

    shutil.rmtree(VFRAMES, ignore_errors=True); os.makedirs(VFRAMES)
    # Hướng video sau khi xoay (ffmpeg tự xoay theo metadata của điện thoại)
    probe_png = f"{VFRAMES}/_probe.png"
    subprocess.run(["ffmpeg", "-v", "error", "-ss", str(TRIM_START), "-i", video, "-frames:v", "1", "-y", probe_png],
                   capture_output=True)
    pr = cv2.imread(probe_png)
    if pr is None:
        raise RuntimeError("ffmpeg không đọc được frame nào từ video.")
    vh, vw = pr.shape[:2]
    os.remove(probe_png)
    short = min(FRAME_HEIGHT, vw, vh)                    # cạnh ngắn tối đa = FRAME_HEIGHT
    scale = f"scale=-2:{short}" if vw >= vh else f"scale={short}:-2"

    cand_fps = min(src_fps, max(1.0, TARGET_FRAMES * CAND_PER_PICK / use))
    hdr = st.get("color_transfer") in ("arib-std-b67", "smpte2084")   # video HDR (iPhone mặc định quay HDR)
    tonemap = ("zscale=t=linear:npl=100,format=gbrpf32le,zscale=p=bt709,tonemap=hable:desat=0,"
               "zscale=t=bt709:m=bt709:r=tv,format=yuv420p,")
    print(f"Video: {os.path.basename(video)} | {vw}x{vh} | {src_fps:.1f} fps | {dur:.1f} s"
          f"{' | HDR' if hdr else ''} -> trích ứng viên {cand_fps:.2f} fps, cạnh ngắn {short}px")

    def _extract(vf):
        cmd = ["ffmpeg", "-v", "error", "-ss", str(TRIM_START), "-i", video, "-t", f"{use:.3f}",
               "-vf", vf, "-q:v", "2", "-y", f"{VFRAMES}/c_%06d.jpg"]
        return subprocess.run(cmd, capture_output=True, text=True)

    res = _extract((tonemap if hdr else "") + f"fps={cand_fps:.5f},{scale}")
    if res.returncode != 0 and hdr:
        print("⚠ ffmpeg không hỗ trợ tone-mapping HDR -> trích bình thường (màu có thể hơi nhạt).")
        for f in glob.glob(f"{VFRAMES}/c_*.jpg"): os.remove(f)
        res = _extract(f"fps={cand_fps:.5f},{scale}")
    if res.returncode != 0:
        raise RuntimeError("ffmpeg lỗi khi trích frame:\n" + res.stderr[-600:])
    cands = sorted(glob.glob(f"{VFRAMES}/c_*.jpg"))
    if not cands:
        raise RuntimeError("ffmpeg không tạo ra frame nào (kiểm tra TRIM_START/TRIM_END).")

    sharp = np.array([_sharpness(c) for c in cands])
    k = max(1, math.ceil(len(cands) / TARGET_FRAMES))       # số ứng viên mỗi đoạn
    picks = [g + int(np.argmax(sharp[g:g + k])) for g in range(0, len(cands), k)]
    med = float(np.median(sharp[picks]))
    blurry = [i for i in picks if sharp[i] < MIN_SHARP_REL * med]
    picks = [i for i in picks if sharp[i] >= MIN_SHARP_REL * med]

    dups, keep, last = 0, [], None
    for i in picks:
        t = _thumb(cands[i])
        if DUP_DIFF > 0 and last is not None and float(np.abs(t - last).mean()) < DUP_DIFF:
            dups += 1
            continue
        keep.append(cands[i]); last = t
    keep_set = set(keep)
    for c in cands:
        if c not in keep_set:
            os.remove(c)
    print(f"Ứng viên {len(cands)} -> chọn {len(keep)} frame "
          f"(bỏ {len(blurry)} frame mờ, {dups} frame trùng do đứng yên)")
    if len(keep) < 60:
        print("⚠ Ít frame (<60): video quá ngắn hoặc rung/mờ nhiều. Nên quay chậm hơn, dài hơn (1–3 phút cho 1 phòng).")
    return keep

# ── 1. Lấy danh sách ảnh ─────────────────────────────────────────────────────
from_video = False
if DATA_KIND == "video":
    images, root, from_video = select_video_frames(DATA_SRC), VFRAMES, True
elif DATA_KIND == "folder":
    root = DATA_SRC
else:
    free_gb = shutil.disk_usage(WORK_DIR).free / 1e9
    print(f"Giải nén {os.path.basename(DATA_SRC)} ({os.path.getsize(DATA_SRC)/1e6:.1f} MB) | ổ trống {free_gb:.1f} GB ...")
    extract(DATA_SRC, EXTRACT)
    root = EXTRACT

if not from_video:
    images = list_files(root, IMG_EXT)

if not images:                                        # zip lồng zip -> giải nén thêm 1 lớp
    inner = list_files(root, {".zip", ".7z", ".rar", ".tar", ".gz", ".tgz"})
    for k, a in enumerate(inner):
        print("Giải nén file nén bên trong:", os.path.relpath(a, root))
        extract(a, f"{EXTRACT}_inner{k}")
    if inner:
        root = WORK_DIR
        images = [p for k in range(len(inner)) for p in list_files(f"{EXTRACT}_inner{k}", IMG_EXT)]

if not images:                                        # trong zip là video -> chọn frame như video thường
    vids = list_files(root, VID_EXT)
    if vids:
        print(f"Không có ảnh nhưng có video trong file nén: {os.path.basename(vids[0])}")
        images, root, from_video = select_video_frames(vids[0]), VFRAMES, True

if not images:
    found = Counter(os.path.splitext(f)[1].lower() or "(không đuôi)" for _, _, fs in os.walk(root) for f in fs)
    raise RuntimeError(f"Không tìm thấy ảnh nào trong dữ liệu. Các loại file có trong đó: {dict(found.most_common(10))}")
print(f"Tìm thấy {len(images)} ảnh")

# ── 2. Đọc ảnh (xoay theo EXIF, hỗ trợ HEIC) ─────────────────────────────────
from PIL import Image, ImageOps
if any(os.path.splitext(p)[1].lower() in (".heic", ".heif") for p in images):
    try:
        import pillow_heif
    except ImportError:
        print("Cài pillow-heif để đọc ảnh HEIC ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pillow-heif"], check=False)
        import pillow_heif
    pillow_heif.register_heif_opener()
Image.MAX_IMAGE_PIXELS = None


def oriented_size(p):
    """Kích thước sau khi xoay theo EXIF (đọc header, không giải mã cả ảnh)."""
    try:
        with Image.open(p) as im:
            w, h = im.size
            if im.getexif().get(0x0112, 1) in (5, 6, 7, 8):
                w, h = h, w
            return w, h
    except Exception:
        return None


sizes = {p: oriented_size(p) for p in images}

# Bộ dữ liệu kiểu Mip-NeRF 360 có cùng ảnh ở nhiều độ phân giải (images/, images_2/, images_4/...):
# mỗi tên ảnh chỉ giữ 1 bản có độ phân giải cao nhất, tránh đưa ảnh trùng vào COLMAP.
best = {}
for p in images:
    if sizes[p] is None: continue
    k = os.path.splitext(os.path.basename(p))[0].lower()
    if k not in best or sizes[p][0] * sizes[p][1] > sizes[best[k]][0] * sizes[best[k]][1]:
        best[k] = p
dups = [p for p in images if sizes[p] is not None and best.get(os.path.splitext(os.path.basename(p))[0].lower()) != p]
if dups:
    print(f"Bỏ {len(dups)} ảnh trùng tên (bản độ phân giải thấp hơn, vd {os.path.relpath(dups[0], root)})")
    dup_set = set(dups)
    images = [p for p in images if p not in dup_set]
bad = [p for p, s in sizes.items() if s is None]
for p in bad:
    print("Bỏ qua (không đọc được):", os.path.relpath(p, root))
images = [p for p in images if sizes[p] is not None]
assert images, "Không đọc được ảnh nào (file hỏng hoặc định dạng lạ)."

# COLMAP dùng 1 camera chung (single_camera=1) -> mọi ảnh phải cùng kích thước & cùng hướng.
aspect = {p: round(sizes[p][0] / sizes[p][1], 2) for p in images}
main_aspect, _ = Counter(aspect.values()).most_common(1)[0]
odd = [p for p in images if abs(aspect[p] - main_aspect) > 0.02]
if odd:
    print(f"⚠ Bỏ {len(odd)} ảnh khác hướng/khác tỉ lệ (ảnh chính có tỉ lệ {main_aspect}), ví dụ:",
          [os.path.basename(p) for p in odd[:5]])
odd_set = set(odd)
images = [p for p in images if p not in odd_set]

if MAX_IMAGES and len(images) > MAX_IMAGES:
    idx = np.linspace(0, len(images) - 1, MAX_IMAGES).round().astype(int)
    images = [images[i] for i in sorted(set(idx))]
    print(f"Lấy đều {len(images)} ảnh (MAX_IMAGES = {MAX_IMAGES})")

med_w = float(np.median([sizes[p][0] for p in images]))
med_h = float(np.median([sizes[p][1] for p in images]))
S = int(min(FRAME_HEIGHT, med_w, med_h)) // 2 * 2       # cạnh ngắn tối đa = FRAME_HEIGHT
if med_w >= med_h:
    TH, TW = S, int(round(med_w * S / med_h / 2) * 2)    # ảnh ngang (vd 1600x1200 -> 1440x1080)
else:
    TW, TH = S, int(round(med_h * S / med_w / 2) * 2)    # ảnh dọc  (vd 1080x1920 giữ nguyên)
print(f"Chuẩn hoá {len(images)} ảnh về {TW}x{TH} px")

# ── 3. Ghi ảnh chuẩn hoá + đo độ nét ─────────────────────────────────────────
shutil.rmtree(FRAMES_DIR, ignore_errors=True); os.makedirs(FRAMES_DIR)
tmp, sharp = [], []
for i, p in enumerate(images, 1):
    try:
        with Image.open(p) as im0:
            im = ImageOps.exif_transpose(im0).convert("RGB")
    except Exception as e:
        print("Bỏ qua (lỗi đọc):", os.path.basename(p), e); continue
    if im.size != (TW, TH):
        im = im.resize((TW, TH), Image.LANCZOS)
    arr = np.asarray(im)
    sharp.append(cv2.Laplacian(cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY), cv2.CV_64F).var())
    out = f"{FRAMES_DIR}/tmp_{i:06d}.png"
    im.save(out, compress_level=3)
    tmp.append(out)
    if i % 50 == 0 or i == len(images):
        print(f"   {i}/{len(images)}", end="\r")
print()
assert tmp, "Không ghi được ảnh nào."

# ── 4. Lọc ảnh mờ, đánh số lại 000001.png ... ───────────────────────────────
drop = set()
if BLUR_DROP_PCT > 0 and len(tmp) >= 20 and not from_video:
    thr = np.percentile(sharp, BLUR_DROP_PCT)
    drop = {f for f, s in zip(tmp, sharp) if s < thr}
for f in drop:
    os.remove(f)
keep = [f for f in tmp if f not in drop]
for i, f in enumerate(keep, 1):
    os.rename(f, f"{FRAMES_DIR}/{i:06d}.png")

# Dọn thư mục giải nén tạm (KHÔNG đụng tới thư mục nguồn của bạn)
for d in glob.glob(f"{EXTRACT}*") + [VFRAMES]:
    shutil.rmtree(d, ignore_errors=True)

n_frames = len(glob.glob(f"{FRAMES_DIR}/*.png"))
print(f"✓ {n_frames} {'frame' if from_video else 'ảnh'} {TW}x{TH} (bỏ {len(drop)} ảnh mờ, {len(odd)} ảnh khác hướng, {len(bad)} ảnh lỗi) -> {FRAMES_DIR}")
if n_frames < 30:
    print("⚠ Ít ảnh (<30) -> COLMAP/3DGS dễ thất bại. Nên có 80–300 ảnh, ảnh liền kề chồng lấp ≥ 70%.")


---
## 2. COLMAP (Structure-from-Motion)
Video → `sequential_matcher` (khớp frame liền kề, nhanh và hợp video hơn exhaustive) + loop detection nếu tải được vocab tree. Sau đó **khử méo** sang camera PINHOLE (`gs_data/`), **loại camera đăng ký sai**, và cảnh báo nếu tỉ lệ đăng ký thấp hoặc camera chỉ xoay tại chỗ (nguyên nhân chính gây floater).

In [ ]:
# ── 2. COLMAP: tính vị trí camera + điểm thưa, khử méo, kiểm tra chất lượng ──
# COLMAP_MATCHER (đặt ở cell Setup):
#   "auto"       -> video: sequential (khớp frame liền kề, nhanh, hợp video) + loop detection nếu tải được vocab tree
#                   ảnh rời: exhaustive (khớp mọi cặp)
#   "sequential" / "exhaustive" -> ép dùng 1 kiểu
import os, re, glob, shutil, struct, subprocess, time, urllib.request
import numpy as np

os.environ["QT_QPA_PLATFORM"] = "offscreen"
assert "n_frames" in globals() and n_frames > 0, "Chưa có frame — chạy cell 1A, 1B trước."

DB          = f"{WORK_DIR}/colmap.db"
colmap_imgs = f"{COLMAP_DIR}/images"
sparse_out  = f"{COLMAP_DIR}/sparse"
DATA_DIR    = f"{WORK_DIR}/gs_data"          # thư mục train cho gsplat: images/ (đã khử méo) + sparse/0 (PINHOLE)
VOCAB_TREE  = f"{BASE}/vocab_tree_flickr100K_words32K.bin"


def colmap(*args, quiet=True):
    """Chạy 1 lệnh COLMAP, báo lỗi rõ ràng nếu thất bại (thay cho '!colmap ...' không bắt lỗi)."""
    t0 = time.time()
    r = subprocess.run(["colmap", *map(str, args)], capture_output=True, text=True)
    log = r.stdout + r.stderr
    if r.returncode != 0:
        raise RuntimeError(f"colmap {args[0]} lỗi (mã {r.returncode}):\n{log[-1500:]}")
    if not quiet:
        print(log[-800:])
    print(f"   ✓ colmap {args[0]} ({time.time() - t0:.0f} s)")
    return log


_HELP = {}
def has_opt(cmd, opt):
    """Kiểm tra phiên bản COLMAP đang cài có tuỳ chọn này không (bản apt và bản mới khác nhau)."""
    if cmd not in _HELP:
        r = subprocess.run(["colmap", cmd, "-h"], capture_output=True, text=True)
        _HELP[cmd] = r.stdout + r.stderr
    return opt in _HELP[cmd]


def read_images_bin(path):
    """Đọc images.bin -> {tên ảnh: tâm camera (3,)}."""
    out = {}
    with open(path, "rb") as f:
        n = struct.unpack("<Q", f.read(8))[0]
        for _ in range(n):
            _id, qw, qx, qy, qz, tx, ty, tz, _cam = struct.unpack("<i7di", f.read(64))
            name = b""
            while (c := f.read(1)) != b"\x00":
                name += c
            n2d = struct.unpack("<Q", f.read(8))[0]
            f.seek(24 * n2d, 1)
            R = np.array([[1 - 2 * (qy*qy + qz*qz), 2 * (qx*qy - qw*qz),     2 * (qx*qz + qw*qy)],
                          [2 * (qx*qy + qw*qz),     1 - 2 * (qx*qx + qz*qz), 2 * (qy*qz - qw*qx)],
                          [2 * (qx*qz - qw*qy),     2 * (qy*qz + qw*qx),     1 - 2 * (qx*qx + qy*qy)]])
            out[name.decode("utf-8", "replace")] = -R.T @ np.array([tx, ty, tz])
    return out


def read_points_bin(path):
    """Đọc points3D.bin -> mảng (N, 3)."""
    pts = []
    with open(path, "rb") as f:
        n = struct.unpack("<Q", f.read(8))[0]
        for _ in range(n):
            _id, x, y, z = struct.unpack("<Q3d", f.read(32))
            f.seek(3 + 8, 1)                                  # rgb + error
            tl = struct.unpack("<Q", f.read(8))[0]
            f.seek(8 * tl, 1)
            pts.append((x, y, z))
    return np.array(pts, dtype=np.float64).reshape(-1, 3)


def count_images(d):
    p = f"{d}/images.bin"
    return struct.unpack("<Q", open(p, "rb").read(8))[0] if os.path.exists(p) else 0


# ── Chuẩn bị thư mục sạch ──
shutil.rmtree(COLMAP_DIR, ignore_errors=True)
shutil.rmtree(DATA_DIR, ignore_errors=True)
shutil.copytree(FRAMES_DIR, colmap_imgs)
if os.path.exists(DB): os.remove(DB)
os.makedirs(sparse_out)
is_video = bool(globals().get("from_video", False))
matcher = COLMAP_MATCHER
if matcher == "auto":
    matcher = "sequential" if (is_video and n_frames > 120) else "exhaustive"
print(f"{n_frames} frame | nguồn: {'video' if is_video else 'ảnh'} | matcher: {matcher}")
print(subprocess.run(["colmap", "-h"], capture_output=True, text=True).stdout.splitlines()[0:1])

print("[1/4] Trích đặc trưng SIFT...")
fe = ["feature_extractor", "--database_path", DB, "--image_path", colmap_imgs,
      "--ImageReader.single_camera", 1, "--ImageReader.camera_model", "SIMPLE_RADIAL",
      "--SiftExtraction.use_gpu", 0]
if has_opt("feature_extractor", "max_num_features"):
    fe += ["--SiftExtraction.max_num_features", 8192]
colmap(*fe)

print(f"[2/4] Matching ({matcher})...")
if matcher == "exhaustive":
    colmap("exhaustive_matcher", "--database_path", DB, "--SiftMatching.use_gpu", 0)
else:
    seq = ["sequential_matcher", "--database_path", DB, "--SiftMatching.use_gpu", 0,
           "--SequentialMatching.overlap", 20]
    if has_opt("sequential_matcher", "quadratic_overlap"):
        seq += ["--SequentialMatching.quadratic_overlap", 1]
    loop_ok = False
    try:   # loop detection: nhận ra khi quay vòng lại chỗ cũ -> bớt trôi (drift). Cần vocab tree.
        if not has_opt("sequential_matcher", "vocab_tree_path"):
            raise RuntimeError("bản COLMAP này không có vocab_tree_path")
        if not os.path.exists(VOCAB_TREE):
            urllib.request.urlretrieve("https://demuc.de/colmap/vocab_tree_flickr100K_words32K.bin", VOCAB_TREE)
        colmap(*seq, "--SequentialMatching.loop_detection", 1, "--SequentialMatching.vocab_tree_path", VOCAB_TREE)
        loop_ok = True
    except Exception as e:
        print(f"   (không dùng được loop detection: {str(e).splitlines()[0][:120]}) -> chạy sequential thường")
    if not loop_ok:
        colmap(*seq)

print("[3/4] Dựng cảnh thưa (mapper)...")
colmap("mapper", "--database_path", DB, "--image_path", colmap_imgs, "--output_path", sparse_out)
models = sorted(m for m in os.listdir(sparse_out) if os.path.isdir(f"{sparse_out}/{m}"))
if not models:
    raise RuntimeError("COLMAP không dựng được cảnh. Quay chậm hơn, đủ sáng, đi tịnh tiến (không chỉ xoay tại chỗ), nhiều chi tiết.")
best = max(models, key=lambda m: count_images(f"{sparse_out}/{m}"))
sizes_m = {m: count_images(f"{sparse_out}/{m}") for m in models}
print(f"   Các model: {sizes_m} -> chọn model {best}")
if len(models) > 1:
    print("   ⚠ Cảnh bị tách thành nhiều mảnh: có đoạn video bị mất liên kết (quay quá nhanh / tường trơn / thiếu sáng).")

print("[4/4] Khử méo ống kính -> camera PINHOLE cho gsplat...")
colmap("image_undistorter", "--image_path", colmap_imgs, "--input_path", f"{sparse_out}/{best}",
       "--output_path", DATA_DIR, "--output_type", "COLMAP")
os.makedirs(f"{DATA_DIR}/sparse/0", exist_ok=True)
for f in glob.glob(f"{DATA_DIR}/sparse/*.bin"):
    shutil.move(f, f"{DATA_DIR}/sparse/0/")
for junk in ("stereo",):
    shutil.rmtree(f"{DATA_DIR}/{junk}", ignore_errors=True)
for f in glob.glob(f"{DATA_DIR}/run-*.sh"):
    os.remove(f)

# ── Kiểm tra chất lượng: tỉ lệ đăng ký, camera ngoại lai, quay xoay tại chỗ ──
centers = read_images_bin(f"{DATA_DIR}/sparse/0/images.bin")
pts = read_points_bin(f"{DATA_DIR}/sparse/0/points3D.bin")
C = np.stack(list(centers.values()))
med = np.median(C, axis=0)
d = np.linalg.norm(C - med, axis=1)
p75 = np.percentile(d, 75)
outliers = [n for n, di in zip(centers, d) if di > max(5 * p75, 1e-9)]
if outliers and len(outliers) <= 0.1 * len(centers):
    print(f"⚠ {len(outliers)} camera nằm rất xa phần còn lại (đăng ký sai) -> loại bỏ:", outliers[:5])
    names_txt = f"{WORK_DIR}/outlier_images.txt"
    open(names_txt, "w").write("\n".join(outliers))
    tmp = f"{DATA_DIR}/sparse/_clean"
    try:
        os.makedirs(tmp, exist_ok=True)
        if not has_opt("image_deleter", "image_names_path"):
            raise RuntimeError("bản COLMAP này không có image_deleter --image_names_path")
        colmap("image_deleter", "--input_path", f"{DATA_DIR}/sparse/0", "--output_path", tmp,
               "--image_names_path", names_txt)
        for f in glob.glob(f"{tmp}/*.bin"):
            shutil.move(f, f"{DATA_DIR}/sparse/0/{os.path.basename(f)}")
        centers = read_images_bin(f"{DATA_DIR}/sparse/0/images.bin")
        C = np.stack(list(centers.values())); med = np.median(C, axis=0); d = np.linalg.norm(C - med, axis=1)
    except Exception as e:
        print("   Không xoá được bằng image_deleter (bước train sẽ tự bù bằng scene scale bền vững):", str(e).splitlines()[0][:150])
    shutil.rmtree(tmp, ignore_errors=True)

n_reg = len(centers)
baseline = float(np.median(d))                                 # độ dịch chuyển điển hình của camera
sample = pts[np.random.default_rng(0).choice(len(pts), min(len(pts), 5000), replace=False)] if len(pts) else pts
depth = float(np.median(np.linalg.norm(sample[:, None, :] - C[None, :: max(1, len(C) // 50)], axis=2))) if len(pts) else 0.0
print(f"\n✓ COLMAP: đăng ký {n_reg}/{n_frames} frame ({100 * n_reg / n_frames:.0f}%), {len(pts):,} điểm thưa")
print(f"  Dịch chuyển camera / khoảng cách tới cảnh = {baseline / max(depth, 1e-9):.3f}")
if n_reg < 0.7 * n_frames:
    print("⚠ Tỉ lệ đăng ký < 70% -> cảnh sẽ thủng/nhiễu. Quay lại chậm hơn, tránh rung, đủ sáng, nhiều chi tiết.")
if depth > 0 and baseline / depth < 0.05:
    print("⚠ Camera gần như KHÔNG DI CHUYỂN (chỉ xoay tại chỗ) -> 3DGS không đo được chiều sâu, sinh nhiều floater.\n"
          "  Hãy vừa đi vừa quay: đi dọc tường, ống kính hướng vào giữa phòng, rồi một vòng ở giữa hướng ra tường.")
print(f"  Dữ liệu train: {DATA_DIR}  (images/ {len(os.listdir(f'{DATA_DIR}/images'))} ảnh + sparse/0)")


---
## 3. Gaussian Splatting (gsplat)
### 3A. Cài gsplat
Thử **v1.5.3** trước (có bilateral grid). Cell tự kiểm tra từng bước: cài đặt → biên dịch CUDA → **chạy thử rasterization trên GPU** → `simple_trainer.py --help` có đủ các cờ cần dùng. Bước nào lỗi thì **tự động lùi về v1.3.0** (chỉ mất bilateral grid, mọi cải tiến khác vẫn chạy). Lý do lỗi được giữ trong biến `errors`.

In [ ]:
# ── 3A. CÀI gsplat + biên dịch CUDA (tự chọn phiên bản chạy được) ───────────
# Thử gsplat 1.5.3 trước (có BILATERAL GRID: bù phơi sáng / cân bằng trắng của điện thoại).
# Nếu 1.5.3 không cài / không biên dịch / không chạy được trên runtime này -> TỰ ĐỘNG lùi về 1.3.0
# (mọi cải tiến khác vẫn chạy, chỉ tắt bilateral grid). Không cần sửa tay.
GSPLAT_CANDIDATES = ["1.5.3", "1.3.0"]
PINNED = "9b9f98a5b440531376b4a5386aea49f8e820203b"   # colmap.py + exif.py dùng pycolmap chính thức (chạy được với numpy 2)

import os, sys, shutil, subprocess, torch

cap = torch.cuda.get_device_capability()
os.environ["TORCH_CUDA_ARCH_LIST"] = f"{cap[0]}.{cap[1]}"
os.environ["MAX_JOBS"] = str(max(1, min(4, (os.cpu_count() or 2) // 2)))
print(f"GPU {torch.cuda.get_device_name(0)} (arch {os.environ['TORCH_CUDA_ARCH_LIST']}) | torch {torch.__version__} | CUDA {torch.version.cuda}")
EX = f"{GSPLAT_REPO}/examples"


def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return r.returncode == 0, (r.stdout + r.stderr)


def need(ok_out, what):
    ok, out = ok_out
    if not ok:
        raise RuntimeError(f"{what} thất bại:\n{out[-2500:]}")
    return out


# Phụ thuộc chung (không phụ thuộc phiên bản)
need(sh('pip install -q tyro viser "imageio[ffmpeg]" tensorboard "torchmetrics[image]" tqdm scipy splines '
        "pycolmap PyYAML piexif scikit-learn tensorly rich matplotlib"), "Cài thư viện phụ")

SSIM_SHIM = '''import torch, torch.nn.functional as F
def _win(c, device, dtype, ws=11, sigma=1.5):
    x = torch.arange(ws, device=device, dtype=torch.float32) - ws // 2
    g = torch.exp(-x ** 2 / (2 * sigma ** 2)); g = g / g.sum()
    return (g[:, None] @ g[None, :]).to(dtype).expand(c, 1, ws, ws).contiguous()
def fused_ssim(img1, img2, padding="same", train=True):
    c = img1.shape[1]; w = _win(c, img1.device, img1.dtype); p = 5 if padding == "same" else 0
    mu1 = F.conv2d(img1, w, padding=p, groups=c); mu2 = F.conv2d(img2, w, padding=p, groups=c)
    s11 = F.conv2d(img1 * img1, w, padding=p, groups=c) - mu1 ** 2
    s22 = F.conv2d(img2 * img2, w, padding=p, groups=c) - mu2 ** 2
    s12 = F.conv2d(img1 * img2, w, padding=p, groups=c) - mu1 * mu2
    C1, C2 = 0.01 ** 2, 0.03 ** 2
    return (((2 * mu1 * mu2 + C1) * (2 * s12 + C2)) / ((mu1 ** 2 + mu2 ** 2 + C1) * (s11 + s22 + C2))).mean()
'''


def setup_version(ver):
    """Cài + biên dịch + kiểm tra 1 phiên bản gsplat. Trả về (True, '') nếu dùng được."""
    print(f"\n── Thử gsplat {ver} ──")
    ok, out = sh(f"pip install -q gsplat=={ver}")
    if not ok: return False, "pip install gsplat:\n" + out
    ok, out = sh("pip install -q " + ("git+https://github.com/nerfstudio-project/nerfview@4538024fe0d15fd1a0e4d760f3695fc44ca72787"
                                      if ver.startswith("1.5") else "nerfview"))
    if not ok: return False, "pip install nerfview:\n" + out
    # mã nguồn examples/ đúng phiên bản
    shutil.rmtree(GSPLAT_REPO, ignore_errors=True)
    ok, out = sh(f"git clone -q --depth 1 -b v{ver} https://github.com/nerfstudio-project/gsplat.git {GSPLAT_REPO}")
    if not ok: return False, "git clone:\n" + out
    open(f"{EX}/datasets/__init__.py", "a").close()
    for rel in ("examples/datasets/colmap.py", "examples/exif.py"):
        ok, out = sh(f"wget -q -O {GSPLAT_REPO}/{rel} https://raw.githubusercontent.com/nerfstudio-project/gsplat/{PINNED}/{rel}")
        if not ok: return False, f"tải {rel}:\n" + out
    if ver.startswith("1.3"):                                   # v1.3.0 đặt tên hàm là 'principle'
        sh(f"sed -i 's/align_principal_axes/align_principle_axes/g' {EX}/datasets/colmap.py")
    if ver.startswith("1.5"):                                   # fused-ssim: biên dịch được thì dùng, không thì dùng bản PyTorch
        sh("pip install -q --no-build-isolation git+https://github.com/rahul-goel/fused-ssim@328dc9836f513d00c4b5bc38fe30478b4435cbb5")
        if not sh(f'{sys.executable} -c "from fused_ssim import fused_ssim"')[0]:
            print("   fused-ssim không biên dịch được -> dùng SSIM PyTorch thuần")
            open(f"{EX}/fused_ssim.py", "w").write(SSIM_SHIM)
    # biên dịch CUDA (JIT) — lần đầu 5–10 phút
    print("   Biên dịch CUDA (5–10 phút lần đầu)...")
    sh("rm -rf ~/.cache/torch_extensions/")
    ok, out = sh(f'{sys.executable} -c "from gsplat.cuda._backend import _C; assert _C is not None; print(\'CUDA_OK\')"')
    if "CUDA_OK" not in out: return False, "biên dịch CUDA:\n" + out
    # chạy thật 1 lần rasterization nhỏ trên GPU (bắt lỗi kernel ngay tại đây)
    test = ("import torch; from gsplat.rendering import rasterization as R; d='cuda'; n=100; "
            "o=R(torch.randn(n,3,device=d)*.3+torch.tensor([0,0,3.],device=d), torch.nn.functional.normalize(torch.randn(n,4,device=d),dim=-1), "
            "torch.rand(n,3,device=d)*.05, torch.rand(n,device=d), torch.rand(n,3,device=d), torch.eye(4,device=d)[None], "
            "torch.tensor([[[50.,0,32],[0,50,32],[0,0,1]]],device=d), 64, 64); print('RASTER_OK', o[0].shape)")
    ok, out = sh(f'{sys.executable} -c "{test}"')
    if "RASTER_OK" not in out: return False, "chạy thử rasterization:\n" + out
    # simple_trainer.py import đủ module + nhận đúng các cờ mà cell train dùng
    ok, out = sh(f"cd {EX} && {sys.executable} simple_trainer.py mcmc --help")
    if not ok: return False, "simple_trainer.py --help:\n" + out
    flags = ["data-dir", "global-scale", "depth-loss", "antialiased", "random-bkgd", "strategy.cap-max", "opacity-reg"]
    miss = [f for f in flags if f not in out and f.replace("-", "_") not in out]
    if miss: return False, f"simple_trainer.py thiếu cờ {miss}"
    open(f"{GSPLAT_REPO}/.version", "w").write(ver)
    return True, ""


GSPLAT_VERSION, errors = None, {}
for ver in GSPLAT_CANDIDATES:
    ok, err = setup_version(ver)
    if ok:
        GSPLAT_VERSION = ver
        break
    errors[ver] = err
    print(f"   ✗ gsplat {ver} không dùng được: {err.splitlines()[0][:150]}")
if GSPLAT_VERSION is None:
    raise RuntimeError("Không phiên bản gsplat nào chạy được trên runtime này. Chi tiết:\n" +
                       "\n\n".join(f"== {v} ==\n{e[-1800:]}" for v, e in errors.items()))

TRAINER_SRC = open(f"{EX}/simple_trainer.py").read()
HAS_BILAGRID = "use_bilateral_grid" in TRAINER_SRC
print(f"\n✓ Dùng gsplat {GSPLAT_VERSION} | bilateral grid: {'CÓ' if HAS_BILAGRID else 'KHÔNG (bản 1.3.0)'}")
if errors:
    print("  (Lý do không dùng bản mới hơn được lưu trong biến `errors` — gửi lại nếu muốn sửa.)")


### 3B. Depth prior (Depth Anything V2)
Mạng ước lượng độ sâu từ 1 ảnh cho biết hình dạng độ sâu ở **mọi pixel** — kể cả tường trơn nơi COLMAP không có điểm. Khi train, loss **Pearson** so sánh hình dạng (bất biến tỉ lệ/dịch) giữa độ sâu render và prior. Cell tự **đo độ tin cậy**: so prior với điểm COLMAP thật của từng ảnh; ảnh nào lệch thì bỏ, cả cảnh không đáng tin thì tắt hẳn (`MONO_OK = False`).

In [ ]:
# ── 3B. DEPTH PRIOR: Depth Anything V2 cho từng ảnh train ──────────────────
# Ý tưởng (DN-Splatter, WACV 2025; loss Pearson như FSGS/SparseGS): mạng ước lượng độ sâu từ 1 ảnh cho biết
# hình dạng độ sâu ở MỌI pixel — kể cả tường trơn nơi COLMAP không có điểm. Loss Pearson so sánh HÌNH DẠNG
# (không cần biết tỉ lệ/độ dịch). Cell này: (1) chạy Depth Anything V2, (2) đo độ tin cậy bằng điểm COLMAP thật
# của từng ảnh, (3) lưu .npy cho bước train. Không đáng tin -> tự TẮT (MONO_OK = False), train vẫn chạy bình thường.
USE_MONO_DEPTH = True
MONO_MODEL     = "depth-anything/Depth-Anything-V2-Small-hf"  # Small: nhanh, Apache-2.0 | "...-Base-hf" đẹp hơn (CC-BY-NC, phi thương mại)
MONO_MIN_CORR  = 0.15     # ảnh có tương quan với điểm COLMAP thấp hơn mức này -> không dùng prior cho ảnh đó
MONO_SCALE     = 0.5      # lưu độ sâu ở 1/2 độ phân giải ảnh train (đủ cho loss, nhẹ RAM)

import os, sys, glob, json, struct, subprocess
import numpy as np, torch
import torch.nn.functional as F
from PIL import Image

# Thư mục examples/datasets của gsplat trùng tên với thư viện HuggingFace `datasets` mà transformers dùng -> gỡ ra
sys.path[:] = [p for p in sys.path if not p.rstrip("/").endswith("/examples")]
for m in [k for k in sys.modules if (k == "datasets" or k.startswith("datasets.")) and "gsplat" in str(getattr(sys.modules[k], "__file__", ""))]:
    del sys.modules[m]

MONO_DIR = f"{DATA_DIR}/mono_depth"
MONO_OK = False
NPARAMS = {0: 3, 1: 4, 2: 4, 3: 5, 4: 8, 5: 8, 6: 12, 7: 5, 8: 4, 9: 5, 10: 12}


def read_colmap_model(d):
    """Đọc cameras.bin, images.bin (kèm điểm 2D), points3D.bin của COLMAP."""
    cams = {}
    with open(f"{d}/cameras.bin", "rb") as f:
        for _ in range(struct.unpack("<Q", f.read(8))[0]):
            cid, model, w, h = struct.unpack("<iiQQ", f.read(24))
            cams[cid] = (model, w, h, np.frombuffer(f.read(8 * NPARAMS[model]), "<f8"))
    ids, xyz = [], []
    with open(f"{d}/points3D.bin", "rb") as f:
        for _ in range(struct.unpack("<Q", f.read(8))[0]):
            pid, x, y, z = struct.unpack("<Q3d", f.read(32)); f.seek(11, 1)
            tl = struct.unpack("<Q", f.read(8))[0]; f.seek(8 * tl, 1)
            ids.append(pid); xyz.append((x, y, z))
    ids, xyz = np.array(ids, np.int64), np.array(xyz, np.float64).reshape(-1, 3)
    order = np.argsort(ids); ids, xyz = ids[order], xyz[order]
    imgs = {}
    with open(f"{d}/images.bin", "rb") as f:
        for _ in range(struct.unpack("<Q", f.read(8))[0]):
            _id, qw, qx, qy, qz, tx, ty, tz, cid = struct.unpack("<i7di", f.read(64))
            name = b""
            while (c := f.read(1)) != b"\x00":
                name += c
            n2d = struct.unpack("<Q", f.read(8))[0]
            p2d = np.frombuffer(f.read(24 * n2d), dtype=[("x", "<f8"), ("y", "<f8"), ("id", "<i8")])
            R = np.array([[1 - 2 * (qy*qy + qz*qz), 2 * (qx*qy - qw*qz),     2 * (qx*qz + qw*qy)],
                          [2 * (qx*qy + qw*qz),     1 - 2 * (qx*qx + qz*qz), 2 * (qy*qz - qw*qx)],
                          [2 * (qx*qz - qw*qy),     2 * (qy*qz + qw*qx),     1 - 2 * (qx*qx + qy*qy)]])
            imgs[name.decode("utf-8", "replace")] = (R, np.array([tx, ty, tz]), cid, p2d)
    return cams, imgs, ids, xyz


def pearson(a, b):
    a = a - a.mean(); b = b - b.mean()
    return float((a * b).sum() / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))


if not USE_MONO_DEPTH:
    print("Bỏ qua depth prior (USE_MONO_DEPTH = False)")
else:
    try:
        from transformers import pipeline
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers"], check=False)
        from transformers import pipeline
    cams, imgs, pids, pxyz = read_colmap_model(f"{DATA_DIR}/sparse/0")
    names = sorted(imgs)
    os.makedirs(MONO_DIR, exist_ok=True)
    for f in glob.glob(f"{MONO_DIR}/*.npy"):
        os.remove(f)
    print(f"Tải mô hình {MONO_MODEL} ...")
    pipe = pipeline("depth-estimation", model=MONO_MODEL, device=0 if torch.cuda.is_available() else -1)

    corrs, used, flipped, B = {}, 0, 0, 8
    for s in range(0, len(names), B):
        batch = names[s:s + B]
        outs = pipe([Image.open(f"{DATA_DIR}/images/{n}").convert("RGB") for n in batch], batch_size=B)
        for name, out in zip(batch, outs):
            R, t, cid, p2d = imgs[name]
            _, W, H, _ = cams[cid]
            pd = out["predicted_depth"]
            pd = (pd if torch.is_tensor(pd) else torch.as_tensor(np.array(pd))).float().squeeze().cpu()
            h, w = max(8, int(H * MONO_SCALE)), max(8, int(W * MONO_SCALE))
            m = F.interpolate(pd[None, None], size=(h, w), mode="bilinear", align_corners=False)[0, 0].numpy()

            # Độ tin cậy: so với độ sâu thật của các điểm COLMAP quan sát được trong ảnh này
            corr = np.nan
            obs = p2d[p2d["id"] >= 0]
            if len(obs) >= 30:
                k = np.searchsorted(pids, obs["id"]); k = np.clip(k, 0, len(pids) - 1)
                good = pids[k] == obs["id"]
                X = pxyz[k[good]]; z = (X @ R.T + t)[:, 2]
                u = np.clip((obs["x"][good] * MONO_SCALE).astype(int), 0, w - 1)
                v = np.clip((obs["y"][good] * MONO_SCALE).astype(int), 0, h - 1)
                sel = z > 1e-6
                if sel.sum() >= 30:
                    disp = 1.0 / z[sel]
                    corr = pearson(m[v[sel], u[sel]], disp)
                    if corr < 0:                        # mô hình trả "độ sâu" thay vì "nghịch đảo độ sâu"
                        m_inv = 1.0 / np.maximum(m, 1e-6)
                        c2 = pearson(m_inv[v[sel], u[sel]], disp)
                        if c2 > corr:
                            m, corr = m_inv, c2; flipped += 1
            corrs[name] = corr
            if np.isnan(corr) or corr >= MONO_MIN_CORR:
                np.save(f"{MONO_DIR}/{os.path.splitext(name)[0]}.npy", m.astype(np.float16))
                used += 1
        print(f"   {min(s + B, len(names))}/{len(names)}", end="\r")

    c = np.array([v for v in corrs.values() if not np.isnan(v)])
    med = float(np.median(c)) if len(c) else float("nan")
    print(f"\nDepth Anything V2: {used}/{len(names)} ảnh dùng được{f' | {flipped} ảnh đã đảo chiều' if flipped else ''}")
    if len(c):
        print(f"Tương quan với điểm COLMAP: trung vị {med:.2f} | ≥0.5: {100 * (c >= 0.5).mean():.0f}% ảnh | "
              f"<{MONO_MIN_CORR}: {100 * (c < MONO_MIN_CORR).mean():.0f}% ảnh (bị loại)")
    MONO_OK = used > 0 and (np.isnan(med) or med >= 0.3)
    if not MONO_OK:
        print("⚠ Depth prior không đáng tin cho cảnh này -> TẮT ở bước train (tránh làm hỏng hình học).")
    json.dump({k: (None if np.isnan(v) else v) for k, v in corrs.items()}, open(f"{MONO_DIR}/_corr.json", "w"))
    del pipe
    torch.cuda.empty_cache()
print("MONO_OK =", MONO_OK, "| thư mục:", MONO_DIR)


### 3C. Train
5 cải tiến bật/tắt độc lập: scene scale bền vững · tâm cảnh theo camera · **Kill-then-Relocate** · **Bilateral grid** · **Depth prior**.

**Bộ dò NaN**: kiểm tra loss mỗi bước và gradient ở 300 bước đầu (sau đó cứ 50 bước). Vừa xuất hiện NaN là dừng ngay, in thành phần gây lỗi, rồi **tự chạy lại** sau khi tắt đúng thành phần đó (depth prior → bilateral grid → fused-ssim → antialiased). Lỗi `device-side assert` trong MCMC relocate trước đây chính là hậu quả của NaN.

Khi bật bilateral grid, xem thêm **cc_psnr** (PSNR sau bù màu).

In [ ]:
# ── 3C. TRAIN 3DGS: MCMC + Kill-then-Relocate + Bilateral grid + Depth prior ──
# Mỗi cải tiến bật/tắt riêng được -> dùng lại y nguyên cho cell ABLATION ở cuối notebook.
#  (1) Scene scale bền vững (--global_scale): 1 camera sai không còn làm scale tăng hàng triệu lần.
#  (2) Tâm cảnh theo vị trí camera (CENTER_METHOD="poses") — hợp cảnh trong phòng.
#  (3) Kill-then-Relocate: định kỳ hạ opacity Gaussian khổng lồ / hình kim / dính ống kính để MCMC tái phân bổ.
#  (4) Bilateral grid (gsplat ≥1.5): học 1 phép biến đổi màu nhỏ cho TỪNG ảnh train -> hấp thụ việc điện thoại
#      tự đổi phơi sáng / cân bằng trắng, thay vì để mô hình "giải thích" bằng floater và vệt màu.
#  (5) Depth prior (cell 3B): loss Pearson giữa độ sâu render và Depth Anything V2 -> tường/sàn phẳng hơn.
import os, sys, re, glob, json, time, shutil, subprocess
import numpy as np

assert os.path.isdir(f"{DATA_DIR}/images") and os.path.isdir(f"{DATA_DIR}/sparse/0"), "Chưa có dữ liệu — chạy cell COLMAP trước."
TRAINER = f"{GSPLAT_REPO}/examples/simple_trainer.py"
assert os.path.exists(TRAINER), "Không thấy simple_trainer.py — chạy cell cài gsplat trước."

# ═════════════ THAM SỐ (chỉnh ở đây) ═════════════
TEST_EVERY    = 8            # cứ 8 ảnh giữ 1 ảnh để đo PSNR/SSIM/LPIPS (không dùng để train)
ANTIALIASED   = True         # Mip-Splatting 2D filter: bớt răng cưa khi zoom
CENTER_METHOD = "poses"      # "poses" (phòng) | "focus" (vật thể ở giữa)
CAP_MAX       = 1_000_000    # TRẦN số Gaussian (MCMC không bao giờ vượt) -> file .ply tối đa 1 triệu Gaussian. None = tự tính theo số ảnh
USE_KR        = True         # (3) Kill-then-Relocate
USE_BILAGRID  = True         # (4) Bilateral grid — tự tắt nếu gsplat < 1.5
USE_MONO      = True         # (5) Depth prior — tự tắt nếu cell 3B báo không đáng tin (MONO_OK = False)
MONO_LAMBDA   = 0.05         #     trọng số loss Pearson
MONO_STOP     = 0.6          #     tắt depth prior sau 60% số bước (để ảnh thật quyết định chi tiết cuối)
KR = dict(start=3000, every=500, stop_frac=0.7, huge=0.15, needle=15.0, needle_min=0.01, near=0.03, max_frac=0.02)
# ═════════════════════════════════════════════════

n_img = len(os.listdir(f"{DATA_DIR}/images"))
if CAP_MAX is None:
    CAP_MAX = int(np.clip(round(n_img * 6000, -5), 500_000, 3_000_000))

SSIM_SHIM = globals().get("SSIM_SHIM") or """import torch, torch.nn.functional as F
def _win(c, device, dtype, ws=11, sigma=1.5):
    x = torch.arange(ws, device=device, dtype=torch.float32) - ws // 2
    g = torch.exp(-x ** 2 / (2 * sigma ** 2)); g = g / g.sum()
    return (g[:, None] @ g[None, :]).to(dtype).expand(c, 1, ws, ws).contiguous()
def fused_ssim(img1, img2, padding="same", train=True):
    c = img1.shape[1]; w = _win(c, img1.device, img1.dtype); p = 5 if padding == "same" else 0
    mu1 = F.conv2d(img1, w, padding=p, groups=c); mu2 = F.conv2d(img2, w, padding=p, groups=c)
    s11 = F.conv2d(img1 * img1, w, padding=p, groups=c) - mu1 ** 2
    s22 = F.conv2d(img2 * img2, w, padding=p, groups=c) - mu2 ** 2
    s12 = F.conv2d(img1 * img2, w, padding=p, groups=c) - mu1 * mu2
    C1, C2 = 0.01 ** 2, 0.03 ** 2
    return (((2 * mu1 * mu2 + C1) * (2 * s12 + C2)) / ((mu1 ** 2 + mu2 ** 2 + C1) * (s11 + s22 + C2))).mean()
"""

# ── Patch 1: tâm cảnh trong colmap.py (idempotent) ──
cm_file = f"{GSPLAT_REPO}/examples/datasets/colmap.py"
src = open(cm_file).read()
src = re.sub(r"similarity_from_cameras\(\s*camtoworlds\s*(,\s*center_method\s*=\s*\"\w+\"\s*)?\)",
             f'similarity_from_cameras(camtoworlds, center_method="{CENTER_METHOD}")', src)
open(cm_file, "w").write(src)

# ── Patch 2 + 3: Kill-then-Relocate và Depth prior vào simple_trainer.py (idempotent) ──
PATCH_FUNCS = r'''
# [KR-PATCH] Kill-then-Relocate cho MCMC. Cấu hình bằng biến môi trường KR_*.
def _kill_then_relocate(runner, step):
    import os as _os
    if _os.environ.get("KR_ENABLE", "0") != "1":
        return
    strat = runner.cfg.strategy
    every, start, stop = (int(_os.environ.get(k, d)) for k, d in (("KR_EVERY", "500"), ("KR_START", "3000"), ("KR_STOP", "20000")))
    if (step < start or step > stop or step % every != 0 or step % strat.refine_every != 0
            or step <= strat.refine_start_iter or step >= strat.refine_stop_iter):
        return
    with torch.no_grad():
        means = runner.splats["means"]
        s_sorted, _ = torch.sort(torch.exp(runner.splats["scales"]), dim=1)
        smid, smax = s_sorted[:, 1], s_sorted[:, 2]
        S = float(runner.scene_scale)
        huge = smax > float(_os.environ.get("KR_HUGE", "0.15")) * S
        needle = (smax > float(_os.environ.get("KR_NEEDLE_MIN", "0.01")) * S) & (
            smax / smid.clamp_min(1e-12) > float(_os.environ.get("KR_NEEDLE", "15")))
        cams = torch.as_tensor(runner.parser.camtoworlds[:, :3, 3], dtype=means.dtype, device=means.device)
        near_thr = float(_os.environ.get("KR_NEAR", "0.03")) * S
        near = torch.zeros(means.shape[0], dtype=torch.bool, device=means.device)
        for i in range(0, means.shape[0], 262144):
            near[i:i + 262144] = torch.cdist(means[i:i + 262144], cams).min(dim=1).values < near_thr
        alive = torch.sigmoid(runner.splats["opacities"]) > strat.min_opacity
        bad = (huge | needle | near) & alive
        max_kill = int(float(_os.environ.get("KR_MAX_FRAC", "0.02")) * means.shape[0])
        if int(bad.sum()) > max_kill > 0:
            score = torch.where(bad, smax, torch.full_like(smax, -1.0))
            keep_idx = torch.topk(score, max_kill).indices
            bad = torch.zeros_like(bad)
            bad[keep_idx] = True
        n = int(bad.sum())
        if n:
            runner.splats["opacities"].data[bad] = torch.logit(
                torch.tensor(strat.min_opacity * 0.5, device=means.device, dtype=means.dtype))
        print(f"[KR] step {step}: kill {n:,}/{means.shape[0]:,} "
              f"(huge {int(huge.sum()):,} | needle {int(needle.sum()):,} | near-cam {int(near.sum()):,}) -> MCMC relocate",
              flush=True)


# [MONO-PATCH] Depth prior: loss Pearson giữa nghịch đảo độ sâu render và Depth Anything V2 (bất biến tỉ lệ/dịch).
_MONO_CACHE = {}
def _mono_depth_loss(runner, step, data, depth_map, alphas):
    import os as _os
    mono_dir, lam = _os.environ.get("MONO_DIR", ""), float(_os.environ.get("MONO_LAMBDA", "0"))
    if not mono_dir or lam <= 0 or depth_map is None or step > int(_os.environ.get("MONO_STOP", "1000000000")):
        return 0.0
    losses = []
    for b in range(depth_map.shape[0]):
        pidx = int(runner.trainset.indices[int(data["image_id"][b])])
        key = _os.path.splitext(runner.parser.image_names[pidx])[0]
        if key not in _MONO_CACHE:
            p = f"{mono_dir}/{key}.npy"
            _MONO_CACHE[key] = torch.from_numpy(np.load(p).astype(np.float32)) if _os.path.exists(p) else None
        m = _MONO_CACHE[key]
        if m is None:
            continue
        H, W = depth_map.shape[1], depth_map.shape[2]
        m = m.to(depth_map.device)
        if tuple(m.shape) != (H, W):
            m = F.interpolate(m[None, None], size=(H, W), mode="bilinear", align_corners=False)[0, 0]
        d = depth_map[b, ..., 0]
        valid = (d > 1e-3) & (alphas[b, ..., 0] > 0.5) & torch.isfinite(d) & torch.isfinite(m)
        if int(valid.sum()) < 256:
            continue
        x = 1.0 / d[valid].clamp_min(1e-3)
        y = m[valid]
        # chặn pixel cực đoan (dính sát ống kính) để gradient của 1/d không bùng nổ
        sub = x.detach()[:: max(1, x.numel() // 100000)].float()
        q = torch.quantile(sub, torch.tensor([0.02, 0.98], device=sub.device))
        x = x.clamp(float(q[0]), float(q[1]))
        x = (x - x.mean()) / (x.std() + 1e-6)
        y = (y - y.mean()) / (y.std() + 1e-6)
        corr = (x * y).mean()
        if bool(torch.isfinite(corr)):
            losses.append(1.0 - corr)
    if not losses:
        return 0.0
    return lam * torch.stack(losses).mean()


# [GUARD-PATCH] Bắt NaN/Inf ngay khi xuất hiện: in thành phần gây lỗi rồi dừng -> cell train tự chạy lại phù hợp.
def _nan_guard_loss(runner, step, loss, L):
    if bool(torch.isfinite(loss)):
        return
    info = {"step": int(step), "where": "loss"}
    for k in ("l1loss", "ssimloss", "depthloss", "tvloss", "_mono_l"):
        v = L.get(k)
        if torch.is_tensor(v):
            info[k] = float(v.detach())
    for k in ("colors", "pixels", "_mono_depth_map", "alphas"):
        v = L.get(k)
        if torch.is_tensor(v):
            info[k + "_finite"] = bool(torch.isfinite(v).all())
    info["params_finite"] = {k: bool(torch.isfinite(v).all()) for k, v in runner.splats.items()}
    print("[NAN] " + json.dumps(info), flush=True)
    raise FloatingPointError(f"Loss khong huu han o buoc {step}")


def _nan_guard_grads(runner, step):
    if step > 300 and step % 50 != 0:
        return
    bad = [k for k, v in runner.splats.items() if v.grad is not None and not bool(torch.isfinite(v.grad).all())]
    for g in (getattr(runner, "bil_grids", None),):
        if g is not None and any(p.grad is not None and not bool(torch.isfinite(p.grad).all()) for p in g.parameters()):
            bad.append("bil_grids")
    if bad:
        print("[NAN] " + json.dumps({"step": int(step), "where": "grad", "params": bad}), flush=True)
        raise FloatingPointError(f"Gradient khong huu han o buoc {step}: {bad}")

'''
subprocess.run(["git", "-C", GSPLAT_REPO, "checkout", "--", "examples/simple_trainer.py"], capture_output=True)
tr = open(TRAINER).read()
a_kr = ("            elif isinstance(self.cfg.strategy, MCMCStrategy):\n"
        "                self.cfg.strategy.step_post_backward(")
a_d1 = "                colors, depths = renders[..., 0:3], renders[..., 3:4]\n"
a_d2 = "                loss += depthloss * cfg.depth_lambda\n"
a_bw = "            loss.backward()\n"
if "[KR-PATCH]" not in tr and all(tr.count(a) == 1 for a in (a_kr, a_d1, a_d2, a_bw)) and tr.count("\ndef main(") == 1:
    tr = tr.replace(a_kr, a_kr.replace("\n                self.cfg",
                    "\n                _kill_then_relocate(self, step)  # [KR-PATCH]\n                self.cfg"), 1)
    tr = tr.replace(a_d1, a_d1 + "                _mono_depth_map = depths  # [MONO-PATCH]\n", 1)
    tr = tr.replace(a_d2, a_d2 + "                _mono_l = _mono_depth_loss(self, step, data, _mono_depth_map, alphas)  # [MONO-PATCH]\n"
                                 "                loss = loss + _mono_l  # [MONO-PATCH]\n", 1)
    tr = tr.replace(a_bw, "            _nan_guard_loss(self, step, loss, locals())  # [GUARD-PATCH]\n" + a_bw +
                          "            _nan_guard_grads(self, step)  # [GUARD-PATCH]\n", 1)
    tr = tr.replace("\ndef main(", PATCH_FUNCS + "\ndef main(", 1)
    open(TRAINER, "w").write(tr)
    print("✓ simple_trainer.py: đã gắn Kill-then-Relocate + Depth prior + bộ dò NaN")
elif "[KR-PATCH]" not in tr:
    USE_KR = USE_MONO = False
    print("⚠ Không tìm thấy chỗ gắn patch (phiên bản gsplat khác) -> train MCMC thường")
PATCHED = "[KR-PATCH]" in open(TRAINER).read()
HAS_BILAGRID = "use_bilateral_grid" in open(TRAINER).read()
MONO_READY = bool(globals().get("MONO_OK", False)) and os.path.isdir(f"{DATA_DIR}/mono_depth")

# ── (1) Scene scale bền vững ──
sys.path.insert(0, f"{GSPLAT_REPO}/examples")
for m in [k for k in sys.modules if k == "datasets" or k.startswith("datasets.")]:
    del sys.modules[m]
from datasets.colmap import Parser
_p = Parser(data_dir=DATA_DIR, factor=DATA_FACTOR, normalize=True, test_every=TEST_EVERY)
_cam = _p.camtoworlds[:, :3, 3]
_dist = np.linalg.norm(_cam - _cam.mean(axis=0), axis=1)
raw_scale, robust_scale = float(_dist.max()), float(np.percentile(_dist, 90))
GLOBAL_SCALE = robust_scale / raw_scale if raw_scale > 1.5 * robust_scale > 0 else 1.0
del _p
print(f"Ảnh: {n_img} | cap_max {CAP_MAX:,} | steps {MAX_STEPS:,} | scene scale gốc {raw_scale * 1.1:.3f} "
      f"-> bền vững {robust_scale * 1.1:.3f} (global_scale {GLOBAL_SCALE:.4f})")
if raw_scale > 10 * robust_scale:
    print("⚠ Có camera nằm rất xa phần còn lại (COLMAP đăng ký sai). Đã bù scale, nhưng nên kiểm tra lại video.")


def _train_once(result_dir, steps, kr, bilagrid, mono, aa, backup, label):
    """Train 1 lần. Trả về (True, số liệu) hoặc (False, {"nan": thông tin NaN | None, "tail": log cuối})."""
    if os.path.isdir(result_dir) and os.listdir(result_dir):
        old, k = f"{result_dir}_old_{time.strftime('%m%d_%H%M%S')}", 1
        while os.path.exists(old):
            old, k = f"{result_dir}_old_{time.strftime('%m%d_%H%M%S')}_{k}", k + 1
        shutil.move(result_dir, old)
        print("Kết quả cũ được giữ ở:", old)
    os.makedirs(result_dir, exist_ok=True)
    save_every = 5000
    save_steps = [str(s) for s in sorted(set(range(save_every, steps + 1, save_every)) | {steps})]
    eval_steps = [str(s) for s in sorted({min(7000, steps), steps})]
    cmd = [sys.executable, "-u", TRAINER, "mcmc",
           "--data_dir", DATA_DIR, "--result_dir", result_dir,
           "--data_factor", str(DATA_FACTOR), "--test_every", str(TEST_EVERY),
           "--max_steps", str(steps), "--global_scale", f"{GLOBAL_SCALE:.6f}",
           "--strategy.cap-max", str(CAP_MAX), "--strategy.refine-stop-iter", str(int(steps * 0.8)),
           "--opacity_reg", "0.01", "--scale_reg", "0.01",
           "--depth_loss", "--depth_lambda", "0.01",
           "--random_bkgd", "--eval_steps", *eval_steps, "--save_steps", *save_steps, "--disable_viewer"]
    if aa:
        cmd.append("--antialiased")
    if bilagrid:
        cmd.append("--use_bilateral_grid")
    env = dict(os.environ, PYTHONUNBUFFERED="1",
               KR_ENABLE="1" if kr else "0", KR_START=str(KR["start"]), KR_EVERY=str(KR["every"]),
               KR_STOP=str(int(steps * KR["stop_frac"])), KR_HUGE=str(KR["huge"]), KR_NEEDLE=str(KR["needle"]),
               KR_NEEDLE_MIN=str(KR["needle_min"]), KR_NEAR=str(KR["near"]), KR_MAX_FRAC=str(KR["max_frac"]),
               MONO_DIR=f"{DATA_DIR}/mono_depth" if mono else "", MONO_LAMBDA=str(MONO_LAMBDA if mono else 0),
               MONO_STOP=str(int(steps * MONO_STOP)))
    print(f"\n▶ {label or os.path.basename(result_dir)} | KR={'bật' if kr else 'tắt'} | bilateral grid={'bật' if bilagrid else 'tắt'}"
          f" | depth prior={'bật' if mono else 'tắt'} | {steps:,} bước", flush=True)
    print(" ".join(cmd), flush=True)

    backed_up = set()
    def backup_latest_ckpt():
        if not (backup and os.path.isdir("/content/drive/MyDrive")):
            return
        ck = sorted(glob.glob(f"{result_dir}/ckpts/*.pt"), key=lambda p: int(re.search(r"ckpt_(\d+)", p).group(1)))
        if not ck or ck[-1] in backed_up or time.time() - os.path.getmtime(ck[-1]) < 20:
            return
        dst = f"{DRIVE_BACKUP}/ckpts"; os.makedirs(dst, exist_ok=True)
        shutil.copy2(ck[-1], dst)
        for f in glob.glob(f"{dst}/*.pt"):
            if os.path.basename(f) != os.path.basename(ck[-1]):
                os.remove(f)
        for f in glob.glob(f"{result_dir}/stats/*.json"):
            shutil.copy2(f, DRIVE_BACKUP)
        backed_up.add(ck[-1])
        print(f"  ↳ đã sao lưu {os.path.basename(ck[-1])} lên Drive", flush=True)

    t0 = time.time()
    proc = subprocess.Popen(cmd, cwd=GSPLAT_REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, env=env)
    last_pct, last_check, kr_lines, tail, nan_info = None, 0, [], [], None
    for line in proc.stdout:
        line = line.rstrip()
        tail = (tail + [line])[-40:]
        mm = re.search(r"(\d+)%\|", line)
        if mm and ("it/s]" in line or "s/it]" in line):
            if mm.group(1) != last_pct:
                print(line[-200:], flush=True); last_pct = mm.group(1)
        else:
            if line.startswith("[KR]"):
                kr_lines.append(line)
            if line.startswith("[NAN]") and nan_info is None:
                try:
                    nan_info = json.loads(line[6:])
                except Exception:
                    nan_info = {"raw": line}
            print(line[-200:], flush=True)
        if time.time() - last_check > 60:
            last_check = time.time(); backup_latest_ckpt()
    rc = proc.wait()
    if backup:
        time.sleep(25); backup_latest_ckpt()
    if rc != 0:
        return False, {"nan": nan_info, "tail": "\n".join(tail)}

    res = dict(label=label or os.path.basename(result_dir), kr=kr, bilagrid=bilagrid, mono=mono, antialiased=aa,
               minutes=round((time.time() - t0) / 60, 1),
               killed=sum(int(re.search(r"kill ([\d,]+)/", l).group(1).replace(",", "")) for l in kr_lines))
    vals = sorted(glob.glob(f"{result_dir}/stats/val_step*.json"), key=lambda p: int(re.search(r"val_step(\d+)", p).group(1)))
    if vals:
        s = json.load(open(vals[-1]))
        res.update({k: s[k] for k in ("psnr", "ssim", "lpips", "cc_psnr", "cc_ssim", "cc_lpips", "num_GS") if k in s})
    trs = sorted(glob.glob(f"{result_dir}/stats/train_step*.json"), key=lambda p: int(re.search(r"train_step(\d+)", p).group(1)))
    if trs:
        res["mem_GB"] = round(json.load(open(trs[-1])).get("mem", 0), 1)
    return True, res




def _bad(info, k):
    v = info.get(k)
    return isinstance(v, float) and not np.isfinite(v)


def run_training(result_dir, steps=None, kr=True, bilagrid=True, mono=True, antialiased=None, backup=False, label=""):
    """Train 1 cấu hình; nếu gặp NaN thì tự chẩn đoán, tắt đúng thành phần gây lỗi và chạy lại (tối đa 5 lần)."""
    steps = steps or MAX_STEPS
    cfg = dict(kr=kr and PATCHED, bilagrid=bilagrid and HAS_BILAGRID, mono=mono and PATCHED and MONO_READY,
               aa=ANTIALIASED if antialiased is None else antialiased)
    notes = []
    for attempt in range(5):
        ok, out = _train_once(result_dir, steps, cfg["kr"], cfg["bilagrid"], cfg["mono"], cfg["aa"], backup, label)
        if ok:
            out["fallbacks"] = notes
            return out
        info = out["nan"]
        if info is None:
            raise RuntimeError("Training LỖI (không phải NaN) — 40 dòng log cuối:\n" + out["tail"])
        shutil.rmtree(result_dir, ignore_errors=True)
        where = f"bước {info.get('step')} ({'loss' if info.get('where') == 'loss' else 'gradient ' + str(info.get('params'))})"
        ssim_shim = f"{GSPLAT_REPO}/examples/fused_ssim.py"
        if cfg["mono"] and (_bad(info, "_mono_l") or info.get("where") == "grad"):
            cfg["mono"] = False; fix = "tắt depth prior"
        elif cfg["bilagrid"] and (_bad(info, "tvloss") or "bil_grids" in (info.get("params") or [])):
            cfg["bilagrid"] = False; fix = "tắt bilateral grid"
        elif _bad(info, "ssimloss") and not _bad(info, "l1loss") and not os.path.exists(ssim_shim) \
                and "fused_ssim" in open(TRAINER).read():
            open(ssim_shim, "w").write(SSIM_SHIM); fix = "đổi fused-ssim sang SSIM PyTorch thuần"
        elif cfg["mono"]:
            cfg["mono"] = False; fix = "tắt depth prior"
        elif cfg["bilagrid"]:
            cfg["bilagrid"] = False; fix = "tắt bilateral grid"
        elif cfg["aa"]:
            cfg["aa"] = False; fix = "tắt antialiased"
        elif cfg["kr"]:
            cfg["kr"] = False; fix = "tắt Kill-then-Relocate"
        else:
            raise RuntimeError(f"Vẫn NaN ở {where} sau khi đã tắt mọi tuỳ chọn. Chi tiết: {info}\n" + out["tail"])
        notes.append(f"NaN ở {where} -> {fix}")
        print(f"\n⚠ Phát hiện NaN ở {where}. Chi tiết: {json.dumps(info, ensure_ascii=False)[:400]}")
        print(f"↻ Tự chạy lại với: {fix}", flush=True)
    raise RuntimeError("Đã thử 5 lần vẫn lỗi: " + "; ".join(notes))


def show_result(r):
    print("\n" + "=" * 64 + f"\nSỐ LIỆU TRAIN — {r['label']}")
    print(f"Thời gian: {r['minutes']} phút | VRAM đỉnh: {r.get('mem_GB', '?')} GB | #GS: {r.get('num_GS', 0):,}")
    if r["kr"]:
        print(f"Kill-then-Relocate: tái phân bổ tổng {r['killed']:,} Gaussian xấu")
    print(f"Held-out: PSNR {r.get('psnr', float('nan')):.2f} | SSIM {r.get('ssim', float('nan')):.3f} | LPIPS {r.get('lpips', float('nan')):.3f}")
    for n in r.get("fallbacks", []):
        print("Tự xử lý:", n)
    if "cc_psnr" in r:
        print(f"Held-out sau bù màu (cc_*): PSNR {r['cc_psnr']:.2f} | SSIM {r['cc_ssim']:.3f} | LPIPS {r['cc_lpips']:.3f}"
              "  <- dùng số này khi bật bilateral grid (ảnh test không có bộ bù màu riêng)")
    print("=" * 64)


MAIN_RESULT = run_training(GSPLAT_OUTPUT, kr=USE_KR, bilagrid=USE_BILAGRID, mono=USE_MONO, backup=True, label="Bản đầy đủ")
show_result(MAIN_RESULT)
json.dump(MAIN_RESULT, open(f"{GSPLAT_OUTPUT}/run_summary.json", "w"), ensure_ascii=False, indent=1)


---
## 4. 3D Scene — xuất file
7 bước lọc (chỉnh `FILTER`, chạy lại riêng cell này, không cần train lại): opacity thấp → ngoài hộp phòng → sát camera → khổng lồ / hình kim → **thấy rõ ở < 3 ảnh train** (đo bằng trọng số blending thật khi render) → đóng góp không đáng kể → SOR. Xuất `scene.ply`, `scene.splat` và `removed.ply` (để soi cái đã bị bỏ).

In [ ]:
# ── 4. XUẤT FILE: lọc floater rồi ghi scene.ply + scene.splat ──────────────
# Chỉnh FILTER rồi chạy lại riêng cell này — KHÔNG cần train lại.
import os, sys, re, glob, json, shutil
import numpy as np, torch
from scipy.spatial import cKDTree

sys.path.insert(0, f"{GSPLAT_REPO}/examples")
from datasets.colmap import Parser                  # cùng parser + cùng chuẩn hoá như lúc train
from gsplat.rendering import rasterization

SH_C0 = 0.28209479177387814
DEV = "cuda"
os.makedirs(VIEWER_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)

FILTER = dict(
    min_opacity     = 0.01,   # (1) bỏ Gaussian gần như trong suốt
    bbox_pct        = 1.0,    # (2) hộp bao phòng từ điểm COLMAP: bỏ 1% điểm ngoài cùng mỗi phía...
    bbox_margin     = 0.10,   #     ...rồi nới hộp thêm 10% mỗi chiều; Gaussian ngoài hộp bị bỏ
    near_cam_frac   = 0.02,   # (3) bỏ Gaussian sát quỹ đạo camera (< 2% đường chéo hộp)
    huge_frac       = 0.05,   # (4) bỏ Gaussian khổng lồ: trục dài > 5% đường chéo hộp
    needle_ratio    = 20.0,   #     bỏ "kim/vệt": trục dài / trục giữa > 20 ...
    needle_min_frac = 0.005,  #     ... và trục dài > 0.5% đường chéo
    min_views       = 3,      # (5) phải đóng góp rõ vào >= 3 ảnh train (floater thường chỉ hiện ở 1–2 ảnh)
    view_eps        = 1.0,    #     "rõ" = tổng trọng số blending >= 1 pixel trong ảnh đó
    keep_energy     = 0.995,  # (6) giữ nhóm Gaussian chiếm 99.5% tổng đóng góp, bỏ phần đuôi vô hình
    sor_k           = 16,     # (7) Statistical Outlier Removal: k láng giềng ...
    sor_std         = 3.0,    #     ... bỏ điểm có khoảng cách TB > mean + 3*std
    max_drop_frac   = 0.5,    # an toàn: bước lọc nào định bỏ > 50% số còn lại thì bỏ qua bước đó
)
SAVE_REMOVED_PLY = True       # xuất thêm removed.ply để soi trong SuperSplat cái gì đã bị bỏ
CKPT_PATH = None              # None = checkpoint mới nhất (máy hoặc Drive); hoặc đường dẫn .pt cụ thể
MAX_VIEWS = 300               # số ảnh train tối đa dùng để đo đóng góp (bước 5, 6)


def write_splat(path, means, log_scales, quats, sh0, logit_op):
    """Ghi .splat (32 byte / Gaussian) cho web viewer."""
    n = len(means)
    order = np.argsort(-np.exp(log_scales.sum(axis=1)) / (1.0 + np.exp(-logit_op)))
    buf = np.zeros(n, dtype=[("pos", "<f4", 3), ("scale", "<f4", 3), ("rgba", "u1", 4), ("rot", "u1", 4)])
    buf["pos"] = means[order]
    buf["scale"] = np.exp(log_scales[order])
    rgb = 0.5 + SH_C0 * sh0[order]
    a = 1.0 / (1.0 + np.exp(-logit_op[order]))
    buf["rgba"] = (np.concatenate([rgb, a[:, None]], axis=1) * 255).clip(0, 255).astype(np.uint8)
    q = quats[order] / np.linalg.norm(quats[order], axis=1, keepdims=True)
    buf["rot"] = (q * 128 + 128).clip(0, 255).astype(np.uint8)
    buf.tofile(path)


def write_ply(path, means, log_scales, quats, sh0, shN, logit_op):
    """Ghi PLY 3DGS chuẩn (mở bằng SuperSplat, antimatter15, ...)."""
    n = len(means)
    rest = shN.transpose(0, 2, 1).reshape(n, -1) if shN is not None else np.zeros((n, 0), np.float32)
    names = (["x", "y", "z", "nx", "ny", "nz", "f_dc_0", "f_dc_1", "f_dc_2"]
             + [f"f_rest_{i}" for i in range(rest.shape[1])]
             + ["opacity", "scale_0", "scale_1", "scale_2", "rot_0", "rot_1", "rot_2", "rot_3"])
    data = np.concatenate([means, np.zeros((n, 3), np.float32), sh0, rest, logit_op[:, None], log_scales, quats],
                          axis=1).astype("<f4")
    header = ("ply\nformat binary_little_endian 1.0\n" f"element vertex {n}\n"
              + "".join(f"property float {k}\n" for k in names) + "end_header\n")
    with open(path, "wb") as f:
        f.write(header.encode()); f.write(data.tobytes())


# ── Đọc checkpoint ──
if CKPT_PATH is None:
    found = glob.glob(f"{GSPLAT_OUTPUT}/**/ckpts/*.pt", recursive=True)
    if not found and os.path.isdir(f"{DRIVE_BACKUP}/ckpts"):
        found = glob.glob(f"{DRIVE_BACKUP}/ckpts/*.pt")
        print("Không có checkpoint trên máy -> dùng bản sao lưu trên Drive")
    assert found, "Không có checkpoint — kiểm tra log training ở cell trước"
    CKPT_PATH = max(found, key=lambda p: int(re.search(r"ckpt_(\d+)", p).group(1)))
print("Checkpoint:", CKPT_PATH)
sp = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)["splats"]
N        = sp["means"].shape[0]
means    = sp["means"].float().numpy()
scales   = sp["scales"].float().numpy()                 # log-scale
quats    = sp["quats"].float().numpy()                  # (w, x, y, z)
logit_op = sp["opacities"].float().numpy().reshape(-1)
sh0      = sp["sh0"].float().numpy().reshape(N, 3)
shN      = sp["shN"].float().numpy() if "shN" in sp else None
op       = 1.0 / (1.0 + np.exp(-np.clip(logit_op, -30, 30)))
print(f"Gaussians: {N:,} | opacity median {np.nanmedian(op):.3f}")

TEST_EVERY = globals().get("TEST_EVERY", 8)
parser = Parser(data_dir=DATA_DIR, factor=DATA_FACTOR, normalize=True, test_every=TEST_EVERY)
cams = parser.camtoworlds[:, :3, 3]

keep = np.isfinite(means).all(1) & np.isfinite(scales).all(1) & np.isfinite(logit_op)
report = [("NaN/Inf", int((~keep).sum()))]


def apply(name, mask):
    """Áp 1 bước lọc (mask=True là GIỮ); bỏ qua nếu bước đó xoá quá nhiều."""
    global keep
    new = keep & mask
    drop = int(keep.sum() - new.sum())
    if drop > FILTER["max_drop_frac"] * keep.sum():
        report.append((name + " (BỎ QUA: xoá quá nhiều)", drop)); return
    keep = new; report.append((name, drop))


# (1) opacity
apply("opacity thấp", op >= FILTER["min_opacity"])

# (2) hộp bao phòng từ điểm COLMAP (đã chuẩn hoá cùng hệ toạ độ)
lo = np.percentile(parser.points, FILTER["bbox_pct"], axis=0)
hi = np.percentile(parser.points, 100 - FILTER["bbox_pct"], axis=0)
lo, hi = np.minimum(lo, cams.min(0)), np.maximum(hi, cams.max(0))        # hộp luôn chứa cả camera
pad = (hi - lo) * FILTER["bbox_margin"]; lo, hi = lo - pad, hi + pad
diag = float(np.linalg.norm(hi - lo))
apply("ngoài hộp phòng", ((means >= lo) & (means <= hi)).all(1))

# (3) sát camera
d_cam, _ = cKDTree(cams).query(means, k=1)
apply("sát camera", d_cam >= FILTER["near_cam_frac"] * diag)

# (4) khổng lồ / kim
s = np.sort(np.exp(scales), axis=1)
smid, smax = s[:, 1], s[:, 2]
huge = smax > FILTER["huge_frac"] * diag
needle = (smax / np.maximum(smid, 1e-12) > FILTER["needle_ratio"]) & (smax > FILTER["needle_min_frac"] * diag)
apply("khổng lồ", ~huge)
apply("hình kim", ~needle)

# (5)+(6) đóng góp thật vào ảnh train: dC/dcolor_i = tổng trọng số blending của Gaussian i trên ảnh
train_idx = [i for i in range(len(parser.image_names)) if i % TEST_EVERY != 0]
if len(train_idx) > MAX_VIEWS:
    train_idx = [train_idx[j] for j in np.linspace(0, len(train_idx) - 1, MAX_VIEWS).round().astype(int)]
idx_t = torch.from_numpy(np.nonzero(keep)[0]).to(DEV)
g_means = torch.from_numpy(means).to(DEV)[idx_t]
g_quats = torch.from_numpy(quats).to(DEV)[idx_t]
g_scales = torch.exp(torch.from_numpy(scales).to(DEV)[idx_t])
g_op = torch.sigmoid(torch.from_numpy(logit_op).to(DEV)[idx_t])
views = torch.zeros(len(idx_t), device=DEV)
energy = torch.zeros(len(idx_t), device=DEV)
mode = "antialiased" if globals().get("ANTIALIASED", True) else "classic"
for i in train_idx:
    cid = parser.camera_ids[i]
    W, H = parser.imsize_dict[cid]
    K = torch.from_numpy(parser.Ks_dict[cid]).float().to(DEV)[None]
    viewmat = torch.linalg.inv(torch.from_numpy(parser.camtoworlds[i]).float().to(DEV))[None]
    colors = torch.ones(len(idx_t), 3, device=DEV, requires_grad=True)
    img, _, _ = rasterization(g_means, g_quats, g_scales, g_op, colors, viewmat, K, W, H,
                              packed=False, rasterize_mode=mode)
    img.sum().backward()
    w = colors.grad.mean(dim=1)                                # tổng trọng số (pixel) trong ảnh này
    views += (w >= FILTER["view_eps"]).float(); energy += w
views, energy = views.cpu().numpy(), energy.cpu().numpy()
sub_views = np.zeros(N); sub_energy = np.zeros(N)
sub_views[keep], sub_energy[keep] = views, energy
apply(f"thấy ở < {FILTER['min_views']} ảnh", sub_views >= FILTER["min_views"])
order = np.argsort(-sub_energy[keep]); cum = np.cumsum(sub_energy[keep][order])
n_keep = int(np.searchsorted(cum, FILTER["keep_energy"] * cum[-1]) + 1) if cum[-1] > 0 else len(order)
m6 = np.zeros(N, bool); m6[np.nonzero(keep)[0][order[:n_keep]]] = True
apply("đóng góp không đáng kể", m6)

# (7) SOR
pts = means[keep]
dk, _ = cKDTree(pts).query(pts, k=FILTER["sor_k"] + 1)
md = dk[:, 1:].mean(1)
m7 = np.ones(N, bool); m7[np.nonzero(keep)[0]] = md <= md.mean() + FILTER["sor_std"] * md.std()
apply("lơ lửng (SOR)", m7)

print("\nBƯỚC LỌC                               SỐ GAUSSIAN BỊ BỎ")
for name, n in report:
    print(f"  {name:<36} {n:>12,}")
print(f"  {'=> CÒN LẠI':<36} {int(keep.sum()):>12,} / {N:,} ({100 * keep.mean():.1f}%)")

PLY_PATH   = f"{OUT_DIR}/scene.ply"
SPLAT_PATH = f"{VIEWER_DIR}/scene.splat"
sel = lambda a, m: a[m] if a is not None else None
write_ply(PLY_PATH, means[keep], scales[keep], quats[keep], sh0[keep], sel(shN, keep), logit_op[keep])
write_splat(SPLAT_PATH, means[keep], scales[keep], quats[keep], sh0[keep], logit_op[keep])
print(f"\n✓ {PLY_PATH}   ({os.path.getsize(PLY_PATH) / 1e6:.1f} MB)")
print(f"✓ {SPLAT_PATH} ({os.path.getsize(SPLAT_PATH) / 1e6:.1f} MB)")
if SAVE_REMOVED_PLY and (~keep).any():
    rm = ~keep & np.isfinite(means).all(1)
    write_ply(f"{OUT_DIR}/removed.ply", means[rm], scales[rm], quats[rm], sh0[rm], sel(shN, rm), logit_op[rm])
    print(f"✓ {OUT_DIR}/removed.ply  (những gì đã bị lọc — mở trong SuperSplat để kiểm tra)")

stats = sorted(glob.glob(f"{GSPLAT_OUTPUT}/stats/val_step*.json"), key=lambda p: int(re.search(r"val_step(\d+)", p).group(1)))
if stats:
    m = json.load(open(stats[-1]))
    print(f"\nHeld-out ({os.path.basename(stats[-1])}): PSNR {m.get('psnr', float('nan')):.2f} dB | "
          f"SSIM {m.get('ssim', float('nan')):.3f} | LPIPS {m.get('lpips', float('nan')):.3f}")
if os.path.isdir("/content/drive/MyDrive"):
    os.makedirs(DRIVE_BACKUP, exist_ok=True)
    for f in (PLY_PATH, SPLAT_PATH):
        shutil.copy2(f, DRIVE_BACKUP)
    print(f"✓ Đã sao lưu scene.ply + scene.splat lên Drive: {DRIVE_BACKUP}")


### 4B. Lấy file .ply (kết quả cuối)
Đặt tên file theo `SCENE_NAME`, lưu lên Drive, tải về máy, và (tuỳ chọn) xuất thêm bản **lite** nhẹ ~3,5 lần (SH bậc 0) cho web/điện thoại.

In [ ]:
# ── 4B. LẤY FILE .PLY (kết quả cuối) ───────────────────────────────────────
# File .ply chuẩn 3DGS (x,y,z, f_dc, f_rest, opacity, scale, rot) — mở được bằng SuperSplat, Polycam viewer,
# antimatter15, Unity/Unreal plugin 3DGS, ... Cell này: đặt tên theo cảnh, lưu Drive, tải về máy.
DOWNLOAD_TO_PC = True     # True = tải .ply về máy (file lớn: nên dùng bản trên Drive nếu tải chậm)
SAVE_TO_DRIVE  = True     # True = chép .ply lên Drive (DRIVE_BACKUP)
MAKE_LITE      = True     # True = xuất thêm bản nhẹ ~3.5 lần (bỏ màu phụ thuộc góc nhìn, SH bậc 0) cho web/điện thoại

import os, shutil

assert "PLY_PATH" in globals() and os.path.exists(PLY_PATH), "Chưa có scene.ply — chạy cell 4 (xuất file) trước."


def ply_count(path):
    head = open(path, "rb").read(4096)
    return int(head.split(b"element vertex ")[1].split(b"\n")[0])


FINAL_PLY = f"{OUT_DIR}/{SCENE_NAME}.ply"
shutil.copy2(PLY_PATH, FINAL_PLY)
outs = [FINAL_PLY]
if MAKE_LITE:
    LITE_PLY = f"{OUT_DIR}/{SCENE_NAME}_lite.ply"
    write_ply(LITE_PLY, means[keep], scales[keep], quats[keep], sh0[keep], None, logit_op[keep])
    outs.append(LITE_PLY)

print("FILE KẾT QUẢ (.ply):")
for p in outs:
    print(f"  {p}  |  {ply_count(p):,} Gaussian  |  {os.path.getsize(p) / 1e6:.1f} MB")

if SAVE_TO_DRIVE and os.path.isdir("/content/drive/MyDrive"):
    os.makedirs(DRIVE_BACKUP, exist_ok=True)
    for p in outs:
        shutil.copy2(p, DRIVE_BACKUP)
    print(f"✓ Đã lưu lên Drive: {DRIVE_BACKUP}")
elif SAVE_TO_DRIVE:
    print("(Chưa mount Drive -> bỏ qua lưu Drive. Chạy cell mount Drive ở mục 1 nếu muốn.)")

if DOWNLOAD_TO_PC and IN_COLAB:
    from google.colab import files
    for p in outs:
        files.download(p)
    print("↓ Đang tải về máy (xem thanh tải của trình duyệt).")
print("Xem nhanh: kéo thả file .ply vào https://superspl.at/editor")


---
## 5. Web 3D Viewer
Tạo `index.html` (gsplat.js, WebGL2) cạnh `scene.splat`, bật web server nhỏ, rồi hiển thị: ngay trong Colab (iframe) hoặc qua link RunPod.

In [ ]:
VIEWER_HTML = r'''<!doctype html>
<html lang="vi">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>SpatialScene3D - 3D Viewer</title>
<style>
  html, body { margin: 0; height: 100%; background: #111; overflow: hidden;
               font-family: system-ui, sans-serif; color: #eee; }
  #ui { position: fixed; top: 12px; left: 12px; z-index: 10; max-width: 330px;
        background: rgba(0,0,0,.6); padding: 10px 12px; border-radius: 8px; font-size: 13px; }
  #ui button { margin: 4px 4px 0 0; padding: 4px 8px; border: 0; border-radius: 4px;
               background: #2d6cdf; color: #fff; cursor: pointer; }
  #hint { opacity: .7; margin-top: 8px; line-height: 1.4; }
</style>
</head>
<body>
<div id="ui">
  <b>SpatialScene3D</b>
  <div id="status">Đang khởi tạo...</div>
  <div>
    <button data-a="x+">Xoay X +90°</button>
    <button data-a="x-">Xoay X -90°</button>
    <button data-a="y+">Xoay Y +90°</button>
    <button data-a="z+">Xoay Z +90°</button>
    <button data-a="flip">Lật 180°</button>
    <button data-a="reset">Reset</button>
  </div>
  <div id="hint">Chuột trái: xoay · Cuộn: zoom · Chuột phải: dịch.<br>
  Nếu cảnh bị nghiêng/ngược, dùng các nút xoay ở trên.<br>
  Có thể kéo thả file .splat / .ply vào cửa sổ để xem file khác.</div>
</div>

<script type="module">
import * as SPLAT from "https://cdn.jsdelivr.net/npm/gsplat@1.2.9/dist/index.es.js";

const statusEl = document.getElementById("status");
const renderer = new SPLAT.WebGLRenderer();
const scene    = new SPLAT.Scene();
const camera   = new SPLAT.Camera();
const controls = new SPLAT.OrbitControls(camera, renderer.canvas);

let splat = null;
const ang = { x: 0, y: 0, z: 0 };

function applyRotation() {
  if (!splat) return;
  splat.rotation = SPLAT.Quaternion.FromEuler(new SPLAT.Vector3(ang.x, ang.y, ang.z));
}

function onLoaded(s) {
  splat = s;
  applyRotation();
  statusEl.textContent = splat.data.vertexCount.toLocaleString() + " Gaussians";
}

async function loadUrl(url) {
  statusEl.textContent = "Đang tải scene...";
  const s = await SPLAT.Loader.LoadAsync(
    url, scene, (p) => { statusEl.textContent = "Đang tải " + Math.round(p * 100) + "%"; });
  onLoaded(s);
}

async function loadFile(file) {
  scene.reset();
  statusEl.textContent = "Đang đọc " + file.name + "...";
  const prog = (p) => { statusEl.textContent = "Đang đọc " + Math.round(p * 100) + "%"; };
  const s = file.name.toLowerCase().endsWith(".ply")
    ? await SPLAT.PLYLoader.LoadFromFileAsync(file, scene, prog)
    : await SPLAT.Loader.LoadFromFileAsync(file, scene, prog);
  onLoaded(s);
}

document.querySelectorAll("button[data-a]").forEach((b) => {
  b.addEventListener("click", () => {
    const a = b.dataset.a, q = Math.PI / 2;
    if (a === "x+") ang.x += q;
    else if (a === "x-") ang.x -= q;
    else if (a === "y+") ang.y += q;
    else if (a === "z+") ang.z += q;
    else if (a === "flip") ang.x += Math.PI;
    else { ang.x = 0; ang.y = 0; ang.z = 0; }
    applyRotation();
  });
});

window.addEventListener("dragover", (e) => e.preventDefault());
window.addEventListener("drop", async (e) => {
  e.preventDefault();
  const f = e.dataTransfer.files[0];
  if (f) { try { await loadFile(f); } catch (err) { statusEl.textContent = "Lỗi: " + err.message; } }
});

(async () => {
  if (location.protocol === "file:") {
    statusEl.textContent = "Mở bằng file:// - hãy kéo thả scene.splat vào cửa sổ.";
  } else {
    try { await loadUrl("scene.splat"); }
    catch (err) { statusEl.textContent = "Không tải được scene.splat: " + err.message; }
  }
})();

const frame = () => {
  controls.update();
  renderer.resize();
  renderer.render(scene, camera);
  requestAnimationFrame(frame);
};
requestAnimationFrame(frame);
</script>
</body>
</html>
'''
open(f"{VIEWER_DIR}/index.html", "w").write(VIEWER_HTML)

# Đóng gói để tải về xem trên máy khác
shutil.make_archive(f"{OUT_DIR}/viewer", "zip", VIEWER_DIR)
print(f"✓ Viewer: {VIEWER_DIR}/index.html")
print(f"✓ Gói tải về: {OUT_DIR}/viewer.zip  (index.html + scene.splat)")

# Bật web server (port 8000)
PORT = 8000
subprocess.run(["pkill", "-f", f"http.server {PORT}"], stderr=subprocess.DEVNULL)
subprocess.Popen([sys.executable, "-m", "http.server", str(PORT), "--bind", "0.0.0.0", "--directory", VIEWER_DIR],
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("\n" + "=" * 70)
if IN_COLAB:
    from google.colab import output, files
    print("Viewer hiển thị ngay bên dưới (cần WebGL2; nếu trống, dùng cách tải về).")
    print("Tải về để xem/nộp: viewer.zip = index.html + scene.splat")
    print("=" * 70)
    output.serve_kernel_port_as_iframe(PORT, path="/index.html", height=620)
    # files.download(f"{OUT_DIR}/viewer.zip")   # bỏ comment để tải về máy
    # files.download(PLY_PATH)
else:
    pod = os.environ.get("RUNPOD_POD_ID")
    if pod:
        print(f"Mở trên trình duyệt:  https://{pod}-{PORT}.proxy.runpod.net/")
        print(f"(cần bật 'Expose HTTP Ports' = {PORT} trong Edit Pod)")
    else:
        print(f"Mở trên trình duyệt:  http://localhost:{PORT}/")
    print("Cách 2: tải viewer.zip về máy, giải nén, chạy  python -m http.server  rồi mở http://localhost:8000")
    print("        hoặc mở index.html bằng double-click rồi kéo thả scene.splat vào cửa sổ.")
    print("=" * 70)


---
## 6. *(Tuỳ chọn)* Ablation cho báo cáo
Train lại cùng dữ liệu với từng tổ hợp cải tiến (mặc định 5 cấu hình × 7000 bước) và xuất bảng so sánh PSNR / SSIM / LPIPS / #Gaussian / thời gian ra `ablation.md` + `ablation.csv`. Đặt `RUN_ABLATION = True` để chạy; chạy lại sẽ bỏ qua cấu hình đã xong (an toàn khi Colab ngắt giữa chừng).

In [ ]:
# ── 6. (TUỲ CHỌN) ABLATION — đo đóng góp của TỪNG cải tiến cho báo cáo ─────
# Chạy cùng 1 bộ dữ liệu với các cấu hình khác nhau, cùng số bước, cùng tập ảnh test (1/8 ảnh không dùng để train).
# 5 cấu hình x 7000 bước trên A100 ~ 30–60 phút.
# Kết quả: bảng Markdown (dán thẳng vào báo cáo) + CSV, lưu ở OUT_DIR và Drive. Chạy lại cell sẽ BỎ QUA cấu hình đã xong.
RUN_ABLATION = False          # đổi thành True để chạy
ABL_STEPS    = MAX_STEPS   # cùng số bước với bản chính (7000)
ABL_CONFIGS  = [              # (tên, KR, bilateral grid, depth prior)
    ("B0 · MCMC (chỉ scene scale bền vững)", False, False, False),
    ("B1 · + Kill-then-Relocate",            True,  False, False),
    ("B2 · + Bilateral grid",                True,  True,  False),
    ("B3 · + Depth prior (đầy đủ)",          True,  True,  True),
    ("B4 · Đầy đủ trừ Kill-then-Relocate",   False, True,  True),
]

import os, json, csv, glob, re, shutil
ABL_ROOT = f"{WORK_DIR}/ablation"
if not RUN_ABLATION:
    print("Bỏ qua ablation (đặt RUN_ABLATION = True để chạy).")
else:
    os.makedirs(ABL_ROOT, exist_ok=True)
    results = []
    for i, (name, kr, bg, mono) in enumerate(ABL_CONFIGS):
        rdir = f"{ABL_ROOT}/B{i}"
        done = f"{rdir}/run_summary.json"
        if os.path.exists(done):
            r = json.load(open(done)); print(f"✓ Đã có: {name}")
        else:
            r = run_training(rdir, steps=ABL_STEPS, kr=kr, bilagrid=bg, mono=mono, label=name)
            json.dump(r, open(done, "w"), ensure_ascii=False, indent=1)
            for ck in sorted(glob.glob(f"{rdir}/ckpts/*.pt"), key=lambda p: int(re.search(r"ckpt_(\d+)", p).group(1)))[:-1]:  # giữ checkpoint cuối
                os.remove(ck)
        results.append(r)

    f = lambda r, k, d: f"{r[k]:.{d}f}" if isinstance(r.get(k), (int, float)) else "—"
    rows = ["| Cấu hình | KR | Bilateral grid | Depth prior | PSNR ↑ | SSIM ↑ | LPIPS ↓ | PSNR bù màu ↑ | #Gaussian | Phút |",
            "|---|:-:|:-:|:-:|--:|--:|--:|--:|--:|--:|"]
    for r in results:
        rows.append(f"| {r['label']} | {'✓' if r['kr'] else ''} | {'✓' if r['bilagrid'] else ''} | {'✓' if r['mono'] else ''} "
                    f"| {f(r, 'psnr', 2)} | {f(r, 'ssim', 3)} | {f(r, 'lpips', 3)} | {f(r, 'cc_psnr', 2)} "
                    f"| {r.get('num_GS', 0):,} | {r['minutes']} |")
    table = "\n".join(rows)
    print("\n" + table)
    md = (f"# Ablation — {SCENE_NAME}\n\n{n_img} ảnh, {ABL_STEPS:,} bước/cấu hình, test = 1/{TEST_EVERY} số ảnh.\n\n{table}\n\n"
          "PSNR bù màu (cc_psnr): PSNR sau khi chỉnh màu ảnh render cho khớp ảnh thật — chỉ có khi bật bilateral grid, "
          "vì ảnh test không có bộ bù màu riêng.\n")
    open(f"{OUT_DIR}/ablation.md", "w").write(md)
    with open(f"{OUT_DIR}/ablation.csv", "w", newline="", encoding="utf-8") as fh:
        keys = ["label", "kr", "bilagrid", "mono", "psnr", "ssim", "lpips", "cc_psnr", "cc_ssim", "cc_lpips", "num_GS", "minutes", "killed"]
        w = csv.DictWriter(fh, fieldnames=keys, extrasaction="ignore"); w.writeheader(); w.writerows(results)
    print(f"\n✓ {OUT_DIR}/ablation.md  ✓ {OUT_DIR}/ablation.csv")
    if os.path.isdir("/content/drive/MyDrive"):
        os.makedirs(DRIVE_BACKUP, exist_ok=True)
        for fn in ("ablation.md", "ablation.csv"):
            shutil.copy2(f"{OUT_DIR}/{fn}", DRIVE_BACKUP)
        print("✓ Đã chép bảng ablation lên Drive:", DRIVE_BACKUP)
